<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/06_07_NeuroFHIR_QC_Longitudinal_and_FHIR_Evidence_Combined_R4_URN_FIXED_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/06_07_NeuroFHIR_QC_Longitudinal_and_FHIR_Evidence_Combined_R4_URN_FIXED_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Combined Notebooks 06 + 07 (FHIR R4 + Transaction URN Corrected)
## Stable-context reconciliation, fresh longitudinal analysis, FHIR evidence, validation, write-back, and read-back

Run this notebook **from top to bottom once**.

It removes the stale-report problem by keeping the dependent steps in one runtime:

1. inspect the current stable synthetic source context;
2. apply the already documented synthetic-context realignment only when still necessary;
3. reseed and verify the corrected prior `Observation`;
4. execute Notebook 06 fresh from the canonical project notebook or current GitHub source;
5. require a newly generated `Scenario alignment: 3/3`;
6. immediately execute Notebook 07 in the same run;
7. generate, validate, write, and read back the linked FHIR evidence.

### Integrity boundary

The reconciliation step may revise only the synthetic stable baseline context. It does not change:

- the public MRI;
- the segmentation mask;
- the model-derived follow-up volume;
- the model provenance;
- the Notebook 05 QC score;
- the severe low-confidence challenge.

All current AI results remain preliminary. Human acceptance, rejection, and correction transitions remain for Notebook 08.

### FHIR R4 corrections in this version

- model checkpoint SHA-256 is represented as a `Device.identifier`;
- `Observation.device` links the model `Device` instead of the invalid `Observation.performer → Device`;
- the invalid `DiagnosticReport.performer → Device` and unsupported `DiagnosticReport.note` are removed;
- every transaction Bundle includes the complete five-resource synthetic source context;
- Bundles are validated before write-back and generated resources are validated after their references exist on the server.


### Transaction-reference correction in this version

Every reference to another resource inside the same transaction Bundle is rewritten in the
embedded Bundle copy from `ResourceType/id` to that entry's deterministic `urn:uuid:...`
`fullUrl`. The standalone resources retain normal relative references for read-back comparison.


### Cell 4 dependency correction in this version

The recursive `collect_references()` helper is now defined before Cell 4 uses it to verify
transaction-Bundle URN rewriting.


# Phase A — Idempotent stable-context reconciliation

This phase checks the **current source files**, not the stale Notebook 06 report. If the stable source is already aligned, it makes no numerical change. If it is still misaligned, it backs up the affected synthetic files, applies the documented mathematical realignment, refreshes index metadata, and reseeds the corrected historical synthetic `Observation`.

In [1]:
# Phase A — Inspect and reconcile the stable synthetic context, then reseed it

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
DEMO_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/demo_case_manifest.json"
)
RESOURCE_INDEX_JSON_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/resource_index.json"
)
RESOURCE_INDEX_CSV_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/resource_index.csv"
)
CASE_PACKAGE_ROOT = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/case_packages"
)
SEGMENTATION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/sample_masks/notebook_04/segmentation_case_manifest.json"
)
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

COMBINED_REPAIR_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_07_combined_reconciliation"
)
BACKUP_ROOT = COMBINED_REPAIR_ROOT / "backups"
SERVER_ROOT = COMBINED_REPAIR_ROOT / "server"
RECONCILIATION_AUDIT_PATH = (
    COMBINED_REPAIR_ROOT / "stable_context_reconciliation_audit.json"
)
RECONCILIATION_MD_PATH = (
    PROJECT_ROOT
    / "docs/NOTEBOOK_06_07_COMBINED_RECONCILIATION.md"
)

FHIR_BASE_URL = os.getenv(
    "NEUROFHIR_QC_COMBINED_FHIR_BASE_URL",
    "https://hapi.fhir.org/baseR4",
).rstrip("/")
FHIR_TIMEOUT_SECONDS = int(
    os.getenv("NEUROFHIR_QC_FHIR_TIMEOUT_SECONDS", "60")
)

for folder in (COMBINED_REPAIR_ROOT, BACKUP_ROOT, SERVER_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")
    temporary.replace(path)

def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required_paths = [
    DEMO_MANIFEST_PATH,
    RESOURCE_INDEX_JSON_PATH,
    SEGMENTATION_MANIFEST_PATH,
]
missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists() or path.stat().st_size == 0
]
if missing_paths:
    raise FileNotFoundError(
        "Required source files are missing:\n"
        + "\n".join(f" - {path}" for path in missing_paths)
    )

demo_manifest = load_json(DEMO_MANIFEST_PATH)
resource_index = load_json(RESOURCE_INDEX_JSON_PATH)
segmentation_manifest = load_json(SEGMENTATION_MANIFEST_PATH)

stable_demo = next(
    case
    for case in demo_manifest.get("cases", [])
    if case.get("case_id") == "stable"
)
stable_segmentation = next(
    case
    for case in segmentation_manifest.get("cases", [])
    if case.get("case_id") == "stable"
)
stable_observation_rows = [
    row
    for row in resource_index.get("resources", [])
    if row.get("case_id") == "stable"
    and row.get("resource_type") == "Observation"
]
if len(stable_observation_rows) != 1:
    raise AssertionError(
        "Expected exactly one stable prior Observation index row."
    )

stable_index_row = stable_observation_rows[0]
stable_observation_path = (
    PROJECT_ROOT / stable_index_row["relative_path"]
)
if not stable_observation_path.exists():
    raise FileNotFoundError(stable_observation_path)

stable_observation = load_json(stable_observation_path)
current_baseline_ml = float(
    stable_observation["valueQuantity"]["value"]
)
executed_followup_ml = float(
    stable_segmentation["region_metrics"]["whole_tumor"][
        "predicted_volume_ml"
    ]
)
planned_percent_change = float(
    stable_demo.get("planned_percent_change", 2.82)
)

if min(current_baseline_ml, executed_followup_ml) <= 0:
    raise AssertionError("Stable volumes must be positive.")
if 1.0 + planned_percent_change / 100.0 <= 0:
    raise AssertionError("Invalid planned percentage change.")

current_percent_change = (
    100.0
    * (executed_followup_ml - current_baseline_ml)
    / current_baseline_ml
)
stable_gate_percent = 10.0
repair_required = abs(current_percent_change) >= stable_gate_percent

backup_rows: list[dict[str, Any]] = []
before_baseline_ml = current_baseline_ml

if repair_required:
    revised_baseline_ml = (
        executed_followup_ml
        / (1.0 + planned_percent_change / 100.0)
    )

    timestamp_slug = utc_now().replace(":", "").replace("-", "")
    run_backup_root = BACKUP_ROOT / timestamp_slug
    run_backup_root.mkdir(parents=True, exist_ok=False)

    files_to_backup = [
        DEMO_MANIFEST_PATH,
        RESOURCE_INDEX_JSON_PATH,
        stable_observation_path,
    ]
    if RESOURCE_INDEX_CSV_PATH.exists():
        files_to_backup.append(RESOURCE_INDEX_CSV_PATH)
    case_package_files = (
        sorted(CASE_PACKAGE_ROOT.glob("*.json"))
        if CASE_PACKAGE_ROOT.exists()
        else []
    )
    files_to_backup.extend(case_package_files)

    for source_path in files_to_backup:
        relative_path = source_path.relative_to(PROJECT_ROOT)
        destination = run_backup_root / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, destination)
        source_hash = sha256_file(source_path)
        backup_hash = sha256_file(destination)
        if source_hash != backup_hash:
            raise AssertionError(
                f"Backup checksum failed: {source_path}"
            )
        backup_rows.append(
            {
                "source_relative_path": relative_path.as_posix(),
                "backup_relative_path": destination.relative_to(
                    PROJECT_ROOT
                ).as_posix(),
                "sha256": source_hash,
            }
        )

    repair_utc = utc_now()

    stable_observation["valueQuantity"]["value"] = round(
        revised_baseline_ml,
        6,
    )
    stable_observation.setdefault("note", []).append(
        {
            "time": repair_utc,
            "text": (
                "Synthetic competition-case baseline realigned to the "
                "preserved executed model-derived follow-up. Public imaging, "
                "segmentation, model output and QC evidence were unchanged."
            ),
        }
    )
    stable_observation.setdefault("meta", {}).setdefault(
        "tag",
        [],
    ).append(
        {
            "system": (
                "https://neurofhir-qc.org/fhir/"
                "CodeSystem/data-origin"
            ),
            "code": "synthetic-scenario-realigned",
            "display": "Synthetic scenario realigned",
        }
    )
    write_json(stable_observation_path, stable_observation)

    stable_demo["baseline_volume_ml"] = round(
        revised_baseline_ml,
        6,
    )
    stable_demo[
        "planned_followup_reference_volume_ml"
    ] = round(executed_followup_ml, 6)
    stable_demo["planned_percent_change"] = round(
        planned_percent_change,
        6,
    )
    stable_demo["planned_change_category"] = "stable"
    stable_demo["scenario_realignment"] = {
        "applied": True,
        "applied_utc": repair_utc,
        "method": (
            "baseline = executed model-derived follow-up / "
            "(1 + planned percentage change / 100)"
        ),
        "original_baseline_volume_ml": round(
            before_baseline_ml,
            6,
        ),
        "revised_baseline_volume_ml": round(
            revised_baseline_ml,
            6,
        ),
        "executed_followup_volume_ml": round(
            executed_followup_ml,
            6,
        ),
        "public_image_or_model_output_changed": False,
        "real_patient_data_changed": False,
    }
    write_json(DEMO_MANIFEST_PATH, demo_manifest)

    new_observation_sha = sha256_file(stable_observation_path)
    new_observation_size = stable_observation_path.stat().st_size
    for key in ("sha256", "checksum_sha256", "file_sha256"):
        if key in stable_index_row:
            stable_index_row[key] = new_observation_sha
    for key in ("size_bytes", "file_size_bytes"):
        if key in stable_index_row:
            stable_index_row[key] = new_observation_size
    stable_index_row["synthetic_scenario_realigned"] = True
    write_json(RESOURCE_INDEX_JSON_PATH, resource_index)

    if RESOURCE_INDEX_CSV_PATH.exists():
        with RESOURCE_INDEX_CSV_PATH.open(
            "r",
            newline="",
            encoding="utf-8",
        ) as handle:
            reader = csv.DictReader(handle)
            csv_rows = list(reader)
            fieldnames = list(reader.fieldnames or [])

        csv_match_count = 0
        for row in csv_rows:
            if (
                row.get("case_id") == "stable"
                and row.get("resource_type") == "Observation"
            ):
                for key in (
                    "sha256",
                    "checksum_sha256",
                    "file_sha256",
                ):
                    if key in row:
                        row[key] = new_observation_sha
                for key in ("size_bytes", "file_size_bytes"):
                    if key in row:
                        row[key] = str(new_observation_size)
                csv_match_count += 1

        if csv_match_count != 1:
            raise AssertionError(
                "Expected one stable Observation CSV index row."
            )
        with RESOURCE_INDEX_CSV_PATH.open(
            "w",
            newline="",
            encoding="utf-8",
        ) as handle:
            writer = csv.DictWriter(
                handle,
                fieldnames=fieldnames,
            )
            writer.writeheader()
            writer.writerows(csv_rows)

    def update_existing_stable_fields(value: Any) -> int:
        updates = 0
        if isinstance(value, dict):
            is_stable = (
                value.get("case_id") == "stable"
                or value.get("demo_case") == "stable"
            )
            if is_stable:
                replacements = {
                    "baseline_volume_ml": round(
                        revised_baseline_ml,
                        6,
                    ),
                    "prior_volume_ml": round(
                        revised_baseline_ml,
                        6,
                    ),
                    "planned_followup_reference_volume_ml": round(
                        executed_followup_ml,
                        6,
                    ),
                    "planned_percent_change": round(
                        planned_percent_change,
                        6,
                    ),
                    "planned_change_category": "stable",
                }
                for key, replacement in replacements.items():
                    if key in value:
                        value[key] = replacement
                        updates += 1
            for child in value.values():
                updates += update_existing_stable_fields(child)
        elif isinstance(value, list):
            for child in value:
                updates += update_existing_stable_fields(child)
        return updates

    for package_path in (
        sorted(CASE_PACKAGE_ROOT.glob("*.json"))
        if CASE_PACKAGE_ROOT.exists()
        else []
    ):
        package_payload = load_json(package_path)
        if update_existing_stable_fields(package_payload):
            write_json(package_path, package_payload)

    current_baseline_ml = round(revised_baseline_ml, 6)
else:
    revised_baseline_ml = current_baseline_ml

reloaded_observation = load_json(stable_observation_path)
persisted_baseline_ml = float(
    reloaded_observation["valueQuantity"]["value"]
)
resulting_percent_change = (
    100.0
    * (executed_followup_ml - persisted_baseline_ml)
    / persisted_baseline_ml
)

if abs(resulting_percent_change) >= stable_gate_percent:
    raise AssertionError(
        "Stable source remains outside the engineering stable gate."
    )

# Reseed even when no repair was necessary so the server and local source
# are synchronized in this same combined run.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "requests>=2.32,<3",
    ]
)
import requests

session = requests.Session()
session.headers.update(
    {
        "Accept": "application/fhir+json, application/json",
        "Content-Type": "application/fhir+json",
        "User-Agent": "NeuroFHIR-QC-Combined-06-07",
    }
)

resource_reference = (
    f"{reloaded_observation['resourceType']}/"
    f"{reloaded_observation['id']}"
)
resource_url = f"{FHIR_BASE_URL}/{resource_reference}"

put_response = session.put(
    resource_url,
    json=reloaded_observation,
    timeout=FHIR_TIMEOUT_SECONDS,
)
if not put_response.ok:
    raise RuntimeError(
        "Stable Observation reseed failed: "
        f"{put_response.status_code} {put_response.text[:1000]}"
    )

get_response = session.get(
    resource_url,
    timeout=FHIR_TIMEOUT_SECONDS,
)
if not get_response.ok:
    raise RuntimeError(
        "Stable Observation read-back failed: "
        f"{get_response.status_code} {get_response.text[:1000]}"
    )
server_observation = get_response.json()
server_baseline_ml = float(
    server_observation["valueQuantity"]["value"]
)
if not math.isclose(
    server_baseline_ml,
    persisted_baseline_ml,
    rel_tol=0,
    abs_tol=1e-6,
):
    raise AssertionError(
        "Server read-back does not match the local stable baseline."
    )

write_json(
    SERVER_ROOT / "stable_observation_readback.json",
    server_observation,
)

reconciliation_audit = {
    "project_name": "NeuroFHIR-QC",
    "status": "completed",
    "audited_utc": utc_now(),
    "repair_required_at_start": repair_required,
    "before_baseline_volume_ml": round(
        before_baseline_ml,
        6,
    ),
    "persisted_baseline_volume_ml": round(
        persisted_baseline_ml,
        6,
    ),
    "executed_model_followup_volume_ml": round(
        executed_followup_ml,
        6,
    ),
    "resulting_percent_change": round(
        resulting_percent_change,
        6,
    ),
    "stable_gate_percent": stable_gate_percent,
    "server_reseed_success": True,
    "server_readback_success": True,
    "public_imaging_changed": False,
    "segmentation_changed": False,
    "model_output_changed": False,
    "qc_evidence_changed": False,
    "real_patient_data_changed": False,
    "backup_rows": backup_rows,
}
write_json(RECONCILIATION_AUDIT_PATH, reconciliation_audit)

RECONCILIATION_MD_PATH.write_text(
    (
        "# Combined Notebook 06–07 Stable-Context Reconciliation\n\n"
        f"**Status:** completed  \n"
        f"**Repair required at start:** {repair_required}  \n"
        f"**Stable baseline before:** {before_baseline_ml:.6f} mL  \n"
        f"**Stable baseline used:** {persisted_baseline_ml:.6f} mL  \n"
        f"**Executed follow-up preserved:** {executed_followup_ml:.6f} mL  \n"
        f"**Resulting change:** {resulting_percent_change:+.6f}%  \n\n"
        "The public MRI, segmentation, model output and QC evidence "
        "were not changed.\n"
    ),
    encoding="utf-8",
)

print("=" * 108)
print("✅ Stable source context is aligned and server-synchronized")
print(f"Baseline used: {persisted_baseline_ml:.6f} mL")
print(f"Executed follow-up preserved: {executed_followup_ml:.6f} mL")
print(f"Resulting change: {resulting_percent_change:+.6f}%")
print(f"Repair applied during this run: {repair_required}")
print(f"✅ Reconciliation audit: {RECONCILIATION_AUDIT_PATH}")
print("=" * 108)

Mounted at /content/drive
✅ Stable source context is aligned and server-synchronized
Baseline used: 18.662712 mL
Executed follow-up preserved: 19.189000 mL
Resulting change: +2.819997%
Repair applied during this run: False
✅ Reconciliation audit: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_06_07_combined_reconciliation/stable_context_reconciliation_audit.json


# Phase B — Execute Notebook 06 fresh in this runtime

The combined notebook now loads the canonical Notebook 06 source from Google Drive. If it is not present there, it downloads the current repository version and saves it to the canonical project notebook directory. Every Notebook 06 code cell is then executed sequentially, producing fresh longitudinal, alignment, audit, and manifest artifacts.

In [2]:
# Phase B — Locate or fetch Notebook 06, save it canonically, and execute all of its code cells

import json
from pathlib import Path

NB06_FILENAME = "06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb"
CANONICAL_NB06_PATH = (
    PROJECT_ROOT / "notebooks" / NB06_FILENAME
)
NB06_CANDIDATES = [
    CANONICAL_NB06_PATH,
    PROJECT_ROOT / NB06_FILENAME,
]

nb06_source_path = next(
    (
        path
        for path in NB06_CANDIDATES
        if path.exists() and path.stat().st_size > 0
    ),
    None,
)

if nb06_source_path is None:
    import requests

    raw_url = (
        "https://raw.githubusercontent.com/"
        "SANGHATI23/neurofhir-qc/main/"
        + NB06_FILENAME
    )
    response = requests.get(
        raw_url,
        timeout=FHIR_TIMEOUT_SECONDS,
    )
    if not response.ok:
        raise RuntimeError(
            "Could not locate Notebook 06 in Drive or download it "
            f"from GitHub: HTTP {response.status_code}"
        )
    notebook_06_payload = response.json()
    CANONICAL_NB06_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    write_json(CANONICAL_NB06_PATH, notebook_06_payload)
    nb06_source_path = CANONICAL_NB06_PATH
else:
    notebook_06_payload = load_json(nb06_source_path)
    if nb06_source_path != CANONICAL_NB06_PATH:
        CANONICAL_NB06_PATH.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        write_json(
            CANONICAL_NB06_PATH,
            notebook_06_payload,
        )
        nb06_source_path = CANONICAL_NB06_PATH

if notebook_06_payload.get("nbformat") != 4:
    raise AssertionError("Notebook 06 is not nbformat 4.")
if not notebook_06_payload.get("cells"):
    raise AssertionError("Notebook 06 has no cells.")

code_cells_06 = [
    cell
    for cell in notebook_06_payload["cells"]
    if cell.get("cell_type") == "code"
    and "".join(cell.get("source", [])).strip()
]
if len(code_cells_06) < 8:
    raise AssertionError(
        f"Notebook 06 appears incomplete: {len(code_cells_06)} code cells."
    )

phase06_namespace: dict[str, Any] = {
    "__name__": "__main__",
}

try:
    from IPython import get_ipython
    from IPython.display import display

    phase06_namespace["get_ipython"] = get_ipython
    phase06_namespace["display"] = display
except Exception:
    pass

print("=" * 108)
print(f"📓 Executing Notebook 06 from: {nb06_source_path}")
print(f"📦 Code cells to execute: {len(code_cells_06)}")
print("=" * 108)

for position, cell in enumerate(code_cells_06, start=1):
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)
    first_line = next(
        (
            line.strip()
            for line in source.splitlines()
            if line.strip()
        ),
        f"Code cell {position}",
    )
    print(
        f"\n▶ Notebook 06 code cell "
        f"{position}/{len(code_cells_06)}: {first_line}"
    )
    try:
        exec(
            compile(
                source,
                f"{NB06_FILENAME}::cell_{position}",
                "exec",
            ),
            phase06_namespace,
            phase06_namespace,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Notebook 06 failed in code cell {position}: "
            f"{first_line}"
        ) from exc

print("\n✅ Notebook 06 code cells completed in the combined runtime")

📓 Executing Notebook 06 from: /content/drive/MyDrive/neurofhir-qc/notebooks/06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb
📦 Code cells to execute: 8

▶ Notebook 06 code cell 1/8: # Cell 1 — Mount Drive, load project state, and enforce the Notebook 05 completion gate
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Notebook 05 completion gate passed
✅ Required source manifests and prior FHIR index found
📓 Notebook 06 path: /content/drive/MyDrive/neurofhir-qc/notebooks/06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb
⚠️ Longitudinal thresholds are engineering display parameters, not clinical response criteria
⚠️ Planned case labels will not override executed model-derived measurements

▶ Notebook 06 code cell 2/8: # Cell 2 — Install the lightweight analysis runtime and record versions
python: 3.12.13
pandas: 2.2.2
matplotlib: 3.10.0
✅ CPU-only longitudinal analysis runtime is ready

▶ Notebook 06 code cell 3/8: # 

# Phase C — Fresh Notebook 06 gate

This gate reads only the artifacts generated in **Phase B of this same run**. Notebook 07 starts only when all three demonstration behaviors align and the Notebook 06 safety metrics remain intact.

In [3]:
# Phase C — Require a freshly regenerated 3/3 Notebook 06 alignment before continuing

NB06_AUDIT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis_audit.json"
)
SCENARIO_ALIGNMENT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis/"
      "scenario_alignment_report.json"
)
LONGITUDINAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/sample_biomarkers/notebook_06/"
      "longitudinal_case_manifest.json"
)

for required_path in (
    NB06_AUDIT_PATH,
    SCENARIO_ALIGNMENT_PATH,
    LONGITUDINAL_MANIFEST_PATH,
):
    if not required_path.exists() or required_path.stat().st_size == 0:
        raise FileNotFoundError(required_path)

fresh_nb06_audit = load_json(NB06_AUDIT_PATH)
fresh_alignment = load_json(SCENARIO_ALIGNMENT_PATH)
fresh_longitudinal_manifest = load_json(
    LONGITUDINAL_MANIFEST_PATH
)

if int(fresh_alignment.get("case_count", 0)) != 3:
    raise AssertionError("Fresh alignment report does not contain 3 cases.")
if int(fresh_alignment.get("alignment_pass_count", 0)) != 3:
    failed = [
        (
            f"{row.get('case_id')}: "
            f"{row.get('reason', 'alignment mismatch')}"
        )
        for row in fresh_alignment.get("rows", [])
        if not bool(row.get("alignment_passed"))
    ]
    raise RuntimeError(
        "Fresh Notebook 06 execution did not achieve 3/3 alignment:\n"
        + "\n".join(f" - {item}" for item in failed)
    )
if not bool(fresh_alignment.get("competition_alignment_ready")):
    raise RuntimeError(
        "Fresh Notebook 06 report still has "
        "competition_alignment_ready=false."
    )

fresh_cases = fresh_longitudinal_manifest.get("cases", [])
if len(fresh_cases) != 3:
    raise AssertionError("Fresh longitudinal manifest is incomplete.")
if any(
    case["workflow_state"]["ai_result_status"] != "preliminary"
    for case in fresh_cases
):
    raise AssertionError("A fresh AI result is not preliminary.")
if any(
    case["workflow_state"]["autonomous_finalization_allowed"]
    for case in fresh_cases
):
    raise AssertionError("Autonomous finalization was allowed.")

stable_case = next(
    case for case in fresh_cases
    if case["case_id"] == "stable"
)

print("=" * 108)
print("✅ Fresh Notebook 06 gate passed")
print("✅ Scenario alignment: 3/3 (100.0%)")
print("✅ competition_alignment_ready = true")
print(
    "Stable case: "
    f"{stable_case['prior_reviewed_baseline_volume_ml']:.6f} → "
    f"{stable_case['selected_current_ai_volume_ml']:.6f} mL | "
    f"{stable_case['percent_change']:+.6f}%"
)
print("✅ All results remain preliminary")
print("✅ Autonomous finalization remains blocked")
print("➡️ Continuing immediately to Notebook 07")
print("=" * 108)

✅ Fresh Notebook 06 gate passed
✅ Scenario alignment: 3/3 (100.0%)
✅ competition_alignment_ready = true
Stable case: 18.662712 → 19.189000 mL | +2.819997%
✅ All results remain preliminary
✅ Autonomous finalization remains blocked
➡️ Continuing immediately to Notebook 07


# Phase D — Notebook 07: FHIR evidence and transaction write-back

The cells below are the full Notebook 07 workflow. Because Notebook 06 was regenerated and verified immediately above, Notebook 07 reads fresh artifacts rather than a stale alignment report.

In [4]:
# Cell 1 — Mount Drive and enforce the complete Notebook 06 gate

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import subprocess
import sys
import textwrap
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab and run it there.") from exc

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

NB06_AUDIT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis_audit.json"
)
LONGITUDINAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/sample_biomarkers/notebook_06/longitudinal_case_manifest.json"
)
LONGITUDINAL_RESULTS_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis/longitudinal_results.json"
)
SCENARIO_ALIGNMENT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis/scenario_alignment_report.json"
)
ALIGNMENT_RECOMMENDATIONS_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis/"
      "synthetic_context_alignment_recommendations.json"
)

RESOURCE_INDEX_PATH = (
    PROJECT_ROOT / "data/synthetic_fhir/notebook_01/resource_index.json"
)
SEGMENTATION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/sample_masks/notebook_04/segmentation_case_manifest.json"
)
QC_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/sample_biomarkers/notebook_05/qc_case_manifest.json"
)

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")
    temporary.replace(path)

def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            if isinstance(manifest.get(key), list):
                return manifest[key]
    raise ValueError("Unrecognized notebook_manifest.json structure.")

def normalize_notebook_number(value: Any) -> str:
    match = re.search(r"\d+", str(value))
    return match.group(0).zfill(2) if match else str(value)

def find_notebook_entry(
    manifest: Any,
    number: str,
) -> dict[str, Any]:
    target = number.zfill(2)
    for entry in notebook_entries(manifest):
        candidates = (
            entry.get("number"),
            entry.get("notebook_number"),
            entry.get("id"),
            entry.get("filename"),
        )
        if any(
            normalize_notebook_number(value) == target
            for value in candidates
            if value is not None
        ):
            return entry
    raise KeyError(f"Notebook {target} is missing from the manifest.")

required_inputs = [
    CONFIG_PATH,
    NOTEBOOK_MANIFEST_PATH,
    NB06_AUDIT_PATH,
    LONGITUDINAL_MANIFEST_PATH,
    LONGITUDINAL_RESULTS_PATH,
    SCENARIO_ALIGNMENT_PATH,
    ALIGNMENT_RECOMMENDATIONS_PATH,
    RESOURCE_INDEX_PATH,
    SEGMENTATION_MANIFEST_PATH,
    QC_MANIFEST_PATH,
]
missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists() or path.stat().st_size == 0
]
if missing_inputs:
    raise FileNotFoundError(
        "Notebook 07 prerequisites are incomplete:\n"
        + "\n".join(f" - {path}" for path in missing_inputs)
    )

project_config = load_json(CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
nb06_audit = load_json(NB06_AUDIT_PATH)
longitudinal_manifest = load_json(LONGITUDINAL_MANIFEST_PATH)
longitudinal_results = load_json(LONGITUDINAL_RESULTS_PATH)
scenario_alignment = load_json(SCENARIO_ALIGNMENT_PATH)
alignment_recommendations = load_json(ALIGNMENT_RECOMMENDATIONS_PATH)
resource_index = load_json(RESOURCE_INDEX_PATH)
segmentation_manifest = load_json(SEGMENTATION_MANIFEST_PATH)
qc_manifest = load_json(QC_MANIFEST_PATH)

nb06_entry = find_notebook_entry(notebook_manifest, "06")
nb07_entry = find_notebook_entry(notebook_manifest, "07")

nb06_status = str(
    nb06_entry.get("status", nb06_audit.get("status", ""))
).strip().lower()
if nb06_status not in {"completed", "complete", "passed"}:
    raise RuntimeError(f"Notebook 06 is not complete: {nb06_status!r}")

required_nb06_rates = {
    "source_prior_observation_validation_rate": 1.0,
    "longitudinal_calculation_success_rate": 1.0,
    "preliminary_status_rate": 1.0,
    "autonomous_finalization_block_rate": 1.0,
    "low_confidence_interpretation_withholding_rate": 1.0,
}
nb06_metrics = nb06_audit.get("metrics", {})
failed_rates = [
    key
    for key, expected in required_nb06_rates.items()
    if float(nb06_metrics.get(key, -1)) != expected
]
if failed_rates:
    raise RuntimeError(
        "Notebook 06 evidence gate failed: " + ", ".join(failed_rates)
    )

competition_alignment_ready = bool(
    scenario_alignment.get(
        "competition_alignment_ready",
        nb06_metrics.get("competition_alignment_ready", False),
    )
)
if not competition_alignment_ready:
    mismatches = [
        (
            f"{row.get('case_id')}: "
            f"{row.get('reason', 'alignment mismatch')}"
        )
        for row in scenario_alignment.get("rows", [])
        if not bool(row.get("alignment_passed"))
    ]
    recommendation_count = int(
        alignment_recommendations.get("recommendation_count", 0)
    )
    raise RuntimeError(
        "Notebook 06 is structurally complete, but its competition-alignment "
        "gate is still closed. Notebook 07 must not write FHIR evidence yet.\n"
        + "\n".join(f" - {item}" for item in mismatches)
        + (
            f"\nNon-applied alignment recommendations: "
            f"{recommendation_count}"
        )
        + "\nRegenerate the affected synthetic prior Observation and dependent "
          "audits, reseed the FHIR source context, and rerun Notebook 06."
    )

cases = longitudinal_manifest.get("cases", [])
CASE_ORDER = ("stable", "progression", "low-confidence")
if {case.get("case_id") for case in cases} != set(CASE_ORDER):
    raise AssertionError("Notebook 06 does not contain the locked three cases.")
if int(longitudinal_results.get("case_count", 0)) != 3:
    raise AssertionError("Notebook 06 longitudinal results are incomplete.")
if int(scenario_alignment.get("alignment_pass_count", 0)) != 3:
    raise AssertionError("All three Notebook 06 cases must be aligned.")

NOTEBOOK_FILENAME = nb07_entry.get(
    "filename",
    "07_NeuroFHIR_QC_FHIR_Evidence_and_Writeback.ipynb",
)
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

FHIR_BASE_URL = os.getenv(
    "NEUROFHIR_QC_NOTEBOOK07_FHIR_BASE_URL",
    "https://hapi.fhir.org/baseR4",
).rstrip("/")
FHIR_TIMEOUT_SECONDS = int(
    os.getenv("NEUROFHIR_QC_FHIR_TIMEOUT_SECONDS", "60")
)

# This notebook action is explicit and limited to synthetic resources.
# The deployable application must still keep write-back disabled by default.
ALLOW_SYNTHETIC_FHIR_WRITEBACK = True

OUTPUT_ROOT = (
    PROJECT_ROOT / "submission/fhir_resources/notebook_07"
)
RESOURCE_ROOT = OUTPUT_ROOT / "resources"
CASE_BUNDLE_ROOT = OUTPUT_ROOT / "case_bundles"
SERVER_RESPONSE_ROOT = OUTPUT_ROOT / "server_responses"
VALIDATION_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/validation"
)
READBACK_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/readback"
)
EVAL_ROOT = (
    PROJECT_ROOT / "evaluation/results/notebook_07_fhir_evidence"
)
DOC_ROOT = PROJECT_ROOT / "docs"

FHIR_EVIDENCE_MANIFEST_PATH = (
    OUTPUT_ROOT / "fhir_evidence_manifest.json"
)
MASTER_EVIDENCE_BUNDLE_PATH = (
    OUTPUT_ROOT / "master_evidence_collection_bundle.json"
)
LOCAL_VALIDATION_PATH = EVAL_ROOT / "local_validation_report.json"
SERVER_VALIDATION_PATH = EVAL_ROOT / "server_validation_report.json"
TRANSACTION_REPORT_PATH = EVAL_ROOT / "transaction_writeback_report.json"
READBACK_REPORT_PATH = EVAL_ROOT / "readback_integrity_report.json"
REFERENCE_GRAPH_JSON = EVAL_ROOT / "fhir_reference_graph.json"
REFERENCE_GRAPH_CSV = EVAL_ROOT / "fhir_reference_graph.csv"
NETWORK_LOG_PATH = EVAL_ROOT / "network_request_log.json"
AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence_writeback_audit.json"
)
AUDIT_MD_PATH = (
    DOC_ROOT / "NOTEBOOK_07_FHIR_EVIDENCE_AND_WRITEBACK.md"
)

for folder in (
    OUTPUT_ROOT,
    RESOURCE_ROOT,
    CASE_BUNDLE_ROOT,
    SERVER_RESPONSE_ROOT,
    VALIDATION_ROOT,
    READBACK_ROOT,
    EVAL_ROOT,
    DOC_ROOT,
):
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 104)
print("✅ Notebook 06 completion and competition-alignment gates passed")
print("✅ Three longitudinal cases are supported by executed evidence")
print("✅ Low-confidence interpretation remains withheld")
print(f"🌐 FHIR R4 server: {FHIR_BASE_URL}")
print(f"📓 Notebook 07 target path: {NOTEBOOK_SAVE_PATH}")
print("⚠️ Synthetic demonstration resources only — never send PHI")
print("⚠️ Human review is not executed in this notebook")
print("=" * 104)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Notebook 06 completion and competition-alignment gates passed
✅ Three longitudinal cases are supported by executed evidence
✅ Low-confidence interpretation remains withheld
🌐 FHIR R4 server: https://hapi.fhir.org/baseR4
📓 Notebook 07 target path: /content/drive/MyDrive/neurofhir-qc/notebooks/07_NeuroFHIR_QC_FHIR_Evidence_and_Writeback.ipynb
⚠️ Synthetic demonstration resources only — never send PHI
⚠️ Human review is not executed in this notebook


In [5]:
# Cell 2 — Configure the FHIR R4 HTTP runtime and verify server capabilities

packages = [
    "requests>=2.32,<3",
    "urllib3>=2,<3",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages]
)

import requests
import urllib3

runtime_versions = {
    "python": sys.version.split()[0],
    "requests": requests.__version__,
    "urllib3": urllib3.__version__,
}

session = requests.Session()
session.headers.update(
    {
        "Accept": "application/fhir+json, application/json",
        "Content-Type": "application/fhir+json",
        "User-Agent": (
            "NeuroFHIR-QC-Notebook07/"
            + str(project_config.get("version", "0.1.0"))
        ),
    }
)

network_log: list[dict[str, Any]] = []

def operation_outcome_summary(payload: Any) -> list[dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    if payload.get("resourceType") != "OperationOutcome":
        return []
    summaries = []
    for issue in payload.get("issue", []):
        details = issue.get("details", {})
        diagnostics = issue.get("diagnostics")
        summaries.append(
            {
                "severity": issue.get("severity"),
                "code": issue.get("code"),
                "details": details.get("text"),
                "diagnostics": diagnostics,
                "expression": issue.get("expression", []),
            }
        )
    return summaries

def request_fhir(
    method: str,
    relative_url: str = "",
    *,
    payload: dict[str, Any] | None = None,
    purpose: str,
    retry_read_like: bool = False,
) -> requests.Response:
    url = (
        FHIR_BASE_URL
        if not relative_url
        else f"{FHIR_BASE_URL}/{relative_url.lstrip('/')}"
    )
    attempts = 3 if retry_read_like else 1
    last_response: requests.Response | None = None
    last_error: Exception | None = None

    for attempt in range(1, attempts + 1):
        started = time.perf_counter()
        try:
            response = session.request(
                method=method,
                url=url,
                json=payload,
                timeout=FHIR_TIMEOUT_SECONDS,
            )
            elapsed = time.perf_counter() - started
            last_response = response

            try:
                returned_payload = response.json()
                returned_resource_type = returned_payload.get(
                    "resourceType"
                )
            except Exception:
                returned_payload = None
                returned_resource_type = None

            network_log.append(
                {
                    "timestamp_utc": utc_now(),
                    "method": method.upper(),
                    "url": url,
                    "status_code": response.status_code,
                    "elapsed_seconds": round(elapsed, 6),
                    "purpose": purpose,
                    "attempt": attempt,
                    "returned_resource_type": returned_resource_type,
                }
            )

            if (
                retry_read_like
                and response.status_code
                in {408, 429, 500, 502, 503, 504}
                and attempt < attempts
            ):
                time.sleep(1.5 * attempt)
                continue
            return response
        except requests.RequestException as exc:
            elapsed = time.perf_counter() - started
            last_error = exc
            network_log.append(
                {
                    "timestamp_utc": utc_now(),
                    "method": method.upper(),
                    "url": url,
                    "status_code": None,
                    "elapsed_seconds": round(elapsed, 6),
                    "purpose": purpose,
                    "attempt": attempt,
                    "error": repr(exc),
                }
            )
            if not retry_read_like or attempt == attempts:
                raise
            time.sleep(1.5 * attempt)

    if last_response is not None:
        return last_response
    raise RuntimeError(
        f"FHIR request failed without a response: {last_error!r}"
    )

metadata_response = request_fhir(
    "GET",
    "metadata",
    purpose="Retrieve FHIR CapabilityStatement",
    retry_read_like=True,
)
if not metadata_response.ok:
    raise RuntimeError(
        f"CapabilityStatement request failed: "
        f"{metadata_response.status_code} "
        f"{metadata_response.text[:500]}"
    )

capability_statement = metadata_response.json()
if capability_statement.get("resourceType") != "CapabilityStatement":
    raise AssertionError("Server metadata is not a CapabilityStatement.")

fhir_version = str(capability_statement.get("fhirVersion", ""))
if not fhir_version.startswith("4.0"):
    raise AssertionError(
        f"Notebook 07 requires FHIR R4 / 4.0.x; server reports "
        f"{fhir_version!r}."
    )

required_server_resources = {
    "Patient",
    "Condition",
    "ImagingStudy",
    "Observation",
    "DiagnosticReport",
    "Device",
    "Provenance",
    "Task",
}
resource_capabilities: dict[str, set[str]] = {}

for rest_block in capability_statement.get("rest", []):
    for resource in rest_block.get("resource", []):
        resource_type = resource.get("type")
        interactions = {
            item.get("code")
            for item in resource.get("interaction", [])
            if item.get("code")
        }
        if resource_type:
            resource_capabilities.setdefault(
                resource_type,
                set(),
            ).update(interactions)

missing_resource_support = sorted(
    required_server_resources.difference(resource_capabilities)
)
if missing_resource_support:
    raise RuntimeError(
        "FHIR server does not advertise required resources: "
        + ", ".join(missing_resource_support)
    )

required_interactions = {"read", "update"}
interaction_failures = {
    resource_type: sorted(
        required_interactions.difference(
            resource_capabilities.get(resource_type, set())
        )
    )
    for resource_type in required_server_resources
    if required_interactions.difference(
        resource_capabilities.get(resource_type, set())
    )
}
if interaction_failures:
    raise RuntimeError(
        "Server interaction support is insufficient: "
        + json.dumps(interaction_failures, indent=2)
    )

write_json(
    VALIDATION_ROOT / "capability_statement.json",
    capability_statement,
)
write_json(
    VALIDATION_ROOT / "capability_summary.json",
    {
        "server_base_url": FHIR_BASE_URL,
        "fhir_version": fhir_version,
        "required_resources": sorted(required_server_resources),
        "advertised_interactions": {
            key: sorted(value)
            for key, value in resource_capabilities.items()
            if key in required_server_resources
        },
        "validated_utc": utc_now(),
    },
)

print("=" * 104)
for name, version in runtime_versions.items():
    print(f"{name}: {version}")
print(f"✅ FHIR server reports version {fhir_version}")
print("✅ Required R4 resource types and read/update interactions advertised")
print("=" * 104)

python: 3.12.13
requests: 2.32.4
urllib3: 2.5.0
✅ FHIR server reports version 4.0.1
✅ Required R4 resource types and read/update interactions advertised


In [6]:
# Cell 3 — Define deterministic FHIR R4 builders and transaction helpers

PROJECT_SYSTEM_ROOT = "https://neurofhir-qc.org/fhir"
IDENTIFIER_ROOT = f"{PROJECT_SYSTEM_ROOT}/identifier"
CODE_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/neurofhir-qc"
TAG_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/data-origin"
METHOD_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/measurement-method"
ACTIVITY_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/provenance-activity"

FHIR_ID_PATTERN = re.compile(r"^[A-Za-z0-9\-\.]{1,64}$")
CASE_ORDER = ("stable", "progression", "low-confidence")

def slug(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9\-\.]+", "-", value.strip())
    cleaned = re.sub(r"-+", "-", cleaned).strip("-")
    if not cleaned:
        raise ValueError("FHIR id slug cannot be empty.")
    if len(cleaned) > 64:
        cleaned = cleaned[:64].rstrip("-")
    if not FHIR_ID_PATTERN.fullmatch(cleaned):
        raise ValueError(f"Invalid generated FHIR id: {cleaned!r}")
    return cleaned

def resource_reference(resource: dict[str, Any]) -> str:
    return f"{resource['resourceType']}/{resource['id']}"

def coding(
    system: str,
    code: str,
    display: str,
) -> dict[str, str]:
    return {
        "system": system,
        "code": code,
        "display": display,
    }

def concept(
    system: str,
    code: str,
    display: str,
    *,
    text: str | None = None,
) -> dict[str, Any]:
    payload: dict[str, Any] = {
        "coding": [coding(system, code, display)]
    }
    payload["text"] = text or display
    return payload

def identifier(
    resource_type: str,
    resource_id: str,
) -> dict[str, str]:
    return {
        "system": f"{IDENTIFIER_ROOT}/{resource_type.lower()}",
        "value": resource_id,
    }

def synthetic_meta(*extra_codes: str) -> dict[str, Any]:
    codes = [
        (
            "synthetic-demonstration",
            "Synthetic demonstration resource",
        ),
        (
            "public-deidentified-imaging-context",
            "Public de-identified imaging context",
        ),
        *[
            (code, code.replace("-", " ").title())
            for code in extra_codes
        ],
    ]
    return {
        "tag": [
            coding(TAG_SYSTEM, code, display)
            for code, display in codes
        ]
    }

def quantity(
    value: float,
    unit: str,
    code: str,
) -> dict[str, Any]:
    if not math.isfinite(float(value)):
        raise ValueError("FHIR Quantity value must be finite.")
    return {
        "value": round(float(value), 6),
        "unit": unit,
        "system": "http://unitsofmeasure.org",
        "code": code,
    }

def deterministic_full_url(
    resource_type: str,
    resource_id: str,
) -> str:
    namespace = uuid.UUID("3a179bae-8ee9-4f7f-9a81-faf1c77439fc")
    value = uuid.uuid5(
        namespace,
        f"{resource_type}/{resource_id}",
    )
    return f"urn:uuid:{value}"

segmentation_cases = segmentation_manifest.get("cases", [])
if len(segmentation_cases) != 3:
    raise AssertionError("Notebook 04 segmentation manifest is incomplete.")
model_case = segmentation_cases[0]

MODEL_NAME = str(
    model_case.get(
        "model_bundle_name",
        "MONAI BraTS MRI Segmentation",
    )
)
MODEL_VERSION = str(
    model_case.get("model_bundle_version", "0.5.4")
)
MODEL_REVISION = str(
    model_case.get("model_bundle_revision", "unknown")
)
MODEL_CHECKPOINT_SHA256 = str(
    model_case.get("model_checkpoint_sha256", "unknown")
)
DEVICE_ID = slug(
    f"nqc-monai-brats-segresnet-{MODEL_VERSION.replace('.', '-')}"
)

def build_device(generated_utc: str) -> dict[str, Any]:
    return {
        "resourceType": "Device",
        "id": DEVICE_ID,
        "meta": synthetic_meta(
            "ai-model",
            "research-example-use",
        ),
        "identifier": [
            identifier("Device", DEVICE_ID),
            {
                "system": f"{IDENTIFIER_ROOT}/model-checkpoint-sha256",
                "value": MODEL_CHECKPOINT_SHA256,
            },
        ],
        "status": "active",
        "manufacturer": "MONAI Project",
        "deviceName": [
            {
                "name": MODEL_NAME,
                "type": "model-name",
            },
            {
                "name": "NeuroFHIR-QC segmentation pipeline",
                "type": "user-friendly-name",
            },
        ],
        "modelNumber": MODEL_REVISION[:64],
        "version": [
            {
                "value": MODEL_VERSION,
            }
        ],
        "type": concept(
            CODE_SYSTEM,
            "neuroimaging-segmentation-software",
            "Neuroimaging segmentation software",
        ),
        "note": [
            {
                "time": generated_utc,
                "text": (
                    "Pinned research/example segmentation software used "
                    "to generate NeuroFHIR-QC demonstration evidence; "
                    "not a diagnostic medical device."
                ),
            }
        ],
    }

def build_observation(
    case: dict[str, Any],
    generated_utc: str,
) -> dict[str, Any]:
    case_id = case["case_id"]
    resource_id = slug(f"nqc-{case_id}-current-ai-volume")
    selected_volume = float(case["selected_current_ai_volume_ml"])
    baseline_volume = float(case["prior_reviewed_baseline_volume_ml"])
    absolute_change = float(case["absolute_change_ml"])
    percent_change = float(case["percent_change"])
    qc_score = float(case["qc_score"])

    return {
        "resourceType": "Observation",
        "id": resource_id,
        "meta": synthetic_meta(
            "ai-generated",
            "preliminary",
            "human-review-required",
        ),
        "identifier": [identifier("Observation", resource_id)],
        "status": "preliminary",
        "category": [
            concept(
                "http://terminology.hl7.org/CodeSystem/"
                "observation-category",
                "imaging",
                "Imaging",
            )
        ],
        "code": concept(
            CODE_SYSTEM,
            "whole-brain-tumor-volume",
            "AI-derived whole-tumor volume",
        ),
        "subject": {"reference": case["patient_reference"]},
        "focus": [{"reference": case["condition_reference"]}],
        "effectiveDateTime": case["followup_datetime"],
        "issued": generated_utc,
        "device": {"reference": f"Device/{DEVICE_ID}"},
        "valueQuantity": quantity(selected_volume, "mL", "mL"),
        "bodySite": concept(
            CODE_SYSTEM,
            "brain-tumor-or-lesion",
            "Brain tumor or lesion",
        ),
        "method": concept(
            METHOD_SYSTEM,
            "monai-segresnet-with-neurofhir-qc",
            "MONAI SegResNet with NeuroFHIR-QC processing",
        ),
        "derivedFrom": [
            {"reference": case["followup_imaging_reference"]},
            {"reference": case["prior_observation_reference"]},
        ],
        "component": [
            {
                "code": concept(
                    CODE_SYSTEM,
                    "prior-reviewed-volume",
                    "Prior reviewed tumor volume",
                ),
                "valueQuantity": quantity(
                    baseline_volume,
                    "mL",
                    "mL",
                ),
            },
            {
                "code": concept(
                    CODE_SYSTEM,
                    "absolute-volume-change",
                    "Absolute tumor-volume change",
                ),
                "valueQuantity": quantity(
                    absolute_change,
                    "mL",
                    "mL",
                ),
            },
            {
                "code": concept(
                    CODE_SYSTEM,
                    "percentage-volume-change",
                    "Percentage tumor-volume change",
                ),
                "valueQuantity": quantity(
                    percent_change,
                    "%",
                    "%",
                ),
            },
            {
                "code": concept(
                    CODE_SYSTEM,
                    "engineering-qc-score",
                    "Engineering QC score",
                ),
                "valueQuantity": {
                    "value": round(qc_score, 6),
                    "unit": "score",
                    "system": CODE_SYSTEM,
                    "code": "score",
                },
            },
            {
                "code": concept(
                    CODE_SYSTEM,
                    "engineering-qc-category",
                    "Engineering QC category",
                ),
                "valueCodeableConcept": concept(
                    CODE_SYSTEM,
                    slug(case["qc_category"].lower()),
                    case["qc_category"],
                ),
            },
            {
                "code": concept(
                    CODE_SYSTEM,
                    "longitudinal-display-category",
                    "Longitudinal display category",
                ),
                "valueCodeableConcept": concept(
                    CODE_SYSTEM,
                    slug(case["longitudinal_interpretation"]),
                    case["longitudinal_display_label"],
                ),
            },
            {
                "code": concept(
                    CODE_SYSTEM,
                    "scenario-alignment",
                    "Competition scenario alignment",
                ),
                "valueBoolean": bool(
                    case["scenario_alignment_passed"]
                ),
            },
            {
                "code": concept(
                    CODE_SYSTEM,
                    "synthetic-challenge-output",
                    "Synthetic challenge output",
                ),
                "valueBoolean": bool(
                    case[
                        "selected_current_volume_is_synthetic_challenge"
                    ]
                ),
            },
        ],
        "note": [
            {
                "text": (
                    case["interpretation_boundary"]
                    + " The measurement and QC thresholds are "
                    "engineering demonstration parameters, not "
                    "validated clinical response criteria. This result "
                    "must remain preliminary until explicit human review."
                )
            }
        ],
    }

def build_diagnostic_report(
    case: dict[str, Any],
    observation: dict[str, Any],
    generated_utc: str,
) -> dict[str, Any]:
    case_id = case["case_id"]
    resource_id = slug(f"nqc-{case_id}-ai-imaging-report")
    conclusion = (
        f"{case['longitudinal_display_label']}. "
        f"Engineering QC category: {case['qc_category']}. "
        "AI-assisted research result remains preliminary and "
        "requires explicit human review."
    )
    return {
        "resourceType": "DiagnosticReport",
        "id": resource_id,
        "meta": synthetic_meta(
            "ai-generated",
            "preliminary",
            "human-review-required",
        ),
        "identifier": [identifier("DiagnosticReport", resource_id)],
        "status": "preliminary",
        "category": [
            concept(
                "http://terminology.hl7.org/CodeSystem/v2-0074",
                "RAD",
                "Radiology",
            )
        ],
        "code": concept(
            CODE_SYSTEM,
            "ai-assisted-neuroimaging-volumetry-report",
            "AI-assisted neuroimaging volumetry report",
        ),
        "subject": {"reference": case["patient_reference"]},
        "effectiveDateTime": case["followup_datetime"],
        "issued": generated_utc,
        "imagingStudy": [
            {"reference": case["followup_imaging_reference"]}
        ],
        "result": [{"reference": resource_reference(observation)}],
        "conclusion": conclusion,
        "conclusionCode": [
            concept(
                CODE_SYSTEM,
                slug(case["longitudinal_interpretation"]),
                case["longitudinal_display_label"],
            )
        ],
    }

def build_task(
    case: dict[str, Any],
    observation: dict[str, Any],
    generated_utc: str,
) -> dict[str, Any]:
    case_id = case["case_id"]
    resource_id = slug(f"nqc-{case_id}-human-review-task")
    manual_review = bool(
        case["workflow_state"]["manual_review_required"]
    )
    priority = "urgent" if manual_review else "routine"
    next_action = (
        "Inspect the unstable segmentation and choose reject or "
        "correction-required."
        if manual_review
        else "Inspect the AI-derived measurement and choose an explicit "
             "review outcome."
    )

    return {
        "resourceType": "Task",
        "id": resource_id,
        "meta": synthetic_meta(
            "human-review-queue",
            "ai-result-preliminary",
        ),
        "identifier": [identifier("Task", resource_id)],
        "status": "requested",
        "intent": "order",
        "priority": priority,
        "code": concept(
            CODE_SYSTEM,
            "review-ai-derived-neuroimaging-result",
            "Review AI-derived neuroimaging result",
        ),
        "description": next_action,
        "focus": {"reference": resource_reference(observation)},
        "for": {"reference": case["patient_reference"]},
        "authoredOn": generated_utc,
        "lastModified": generated_utc,
        "requester": {"reference": f"Device/{DEVICE_ID}"},
        "reasonCode": concept(
            CODE_SYSTEM,
            slug(case["qc_category"].lower()),
            case["qc_category"],
        ),
        "input": [
            {
                "type": concept(
                    CODE_SYSTEM,
                    "engineering-qc-score",
                    "Engineering QC score",
                ),
                "valueDecimal": round(float(case["qc_score"]), 6),
            },
            {
                "type": concept(
                    CODE_SYSTEM,
                    "engineering-qc-category",
                    "Engineering QC category",
                ),
                "valueString": case["qc_category"],
            },
            {
                "type": concept(
                    CODE_SYSTEM,
                    "longitudinal-interpretation",
                    "Longitudinal interpretation",
                ),
                "valueString": case["longitudinal_interpretation"],
            },
            {
                "type": concept(
                    CODE_SYSTEM,
                    "scenario-alignment",
                    "Competition scenario alignment",
                ),
                "valueBoolean": bool(
                    case["scenario_alignment_passed"]
                ),
            },
            {
                "type": concept(
                    CODE_SYSTEM,
                    "selected-mask-artifact",
                    "Selected segmentation-mask artifact",
                ),
                "valueString": case["source_artifacts"][
                    "selected_mask_file"
                ],
            },
        ],
        "note": [
            {
                "time": generated_utc,
                "text": (
                    "No reviewer has acted yet. Notebook 08 must record "
                    "acceptance, rejection, or correction-required with "
                    "reviewer role, reason, note, and provenance."
                ),
            }
        ],
    }

def build_provenance(
    case: dict[str, Any],
    observation: dict[str, Any],
    report: dict[str, Any],
    task: dict[str, Any],
    generated_utc: str,
) -> dict[str, Any]:
    case_id = case["case_id"]
    resource_id = slug(f"nqc-{case_id}-ai-generation-provenance")
    return {
        "resourceType": "Provenance",
        "id": resource_id,
        "meta": synthetic_meta(
            "algorithmic-generation",
            "review-not-yet-performed",
        ),
        "target": [
            {"reference": resource_reference(observation)},
            {"reference": resource_reference(report)},
            {"reference": resource_reference(task)},
        ],
        "occurredDateTime": generated_utc,
        "recorded": generated_utc,
        "reason": [
            concept(
                ACTIVITY_SYSTEM,
                "generate-reviewable-neuroimaging-evidence",
                "Generate reviewable neuroimaging evidence",
            )
        ],
        "activity": concept(
            ACTIVITY_SYSTEM,
            "ai-assisted-generation",
            "AI-assisted evidence generation",
        ),
        "agent": [
            {
                "type": concept(
                    "http://terminology.hl7.org/CodeSystem/"
                    "provenance-participant-type",
                    "author",
                    "Author",
                ),
                "who": {"reference": f"Device/{DEVICE_ID}"},
            }
        ],
        "entity": [
            {
                "role": "source",
                "what": {
                    "reference": case[
                        "followup_imaging_reference"
                    ]
                },
            },
            {
                "role": "source",
                "what": {
                    "reference": case[
                        "prior_observation_reference"
                    ]
                },
            },
        ],
        "policy": [
            "https://neurofhir-qc.org/policy/"
            "synthetic-fhir-public-imaging-only"
        ],
    }

def replace_in_bundle_references(
    value: Any,
    full_url_by_reference: dict[str, str],
) -> Any:
    """
    Recursively replace ResourceType/id references with the matching
    Bundle.entry.fullUrl URN when the referenced resource is present in
    the same transaction Bundle.

    The original standalone resources remain unchanged. Only the copies
    embedded in the transaction Bundle use URN references.
    """
    if isinstance(value, dict):
        replaced: dict[str, Any] = {}
        for key, child in value.items():
            if (
                key == "reference"
                and isinstance(child, str)
                and child in full_url_by_reference
            ):
                replaced[key] = full_url_by_reference[child]
            else:
                replaced[key] = replace_in_bundle_references(
                    child,
                    full_url_by_reference,
                )
        return replaced

    if isinstance(value, list):
        return [
            replace_in_bundle_references(
                child,
                full_url_by_reference,
            )
            for child in value
        ]

    return value


def transaction_entry(
    original_resource: dict[str, Any],
    bundled_resource: dict[str, Any],
    full_url: str,
) -> dict[str, Any]:
    resource_type = original_resource["resourceType"]
    resource_id = original_resource["id"]
    return {
        "fullUrl": full_url,
        "resource": bundled_resource,
        "request": {
            "method": "PUT",
            "url": f"{resource_type}/{resource_id}",
        },
    }


def build_transaction_bundle(
    case_id: str,
    resources: Iterable[dict[str, Any]],
    generated_utc: str,
) -> dict[str, Any]:
    resource_list = list(resources)

    references = [
        resource_reference(resource)
        for resource in resource_list
    ]
    if len(references) != len(set(references)):
        duplicates = sorted(
            reference
            for reference in set(references)
            if references.count(reference) > 1
        )
        raise AssertionError(
            f"{case_id}: duplicate resources in transaction Bundle: "
            + ", ".join(duplicates)
        )

    full_url_by_reference = {
        resource_reference(resource): deterministic_full_url(
            resource["resourceType"],
            resource["id"],
        )
        for resource in resource_list
    }

    entries: list[dict[str, Any]] = []
    for resource in resource_list:
        reference = resource_reference(resource)

        # JSON round-trip produces a fully detached copy and avoids mutating
        # the standalone resources used for local validation and read-back.
        bundled_resource = json.loads(
            json.dumps(resource)
        )
        bundled_resource = replace_in_bundle_references(
            bundled_resource,
            full_url_by_reference,
        )

        entries.append(
            transaction_entry(
                original_resource=resource,
                bundled_resource=bundled_resource,
                full_url=full_url_by_reference[reference],
            )
        )

    bundle_id = slug(f"nqc-{case_id}-evidence-transaction")
    return {
        "resourceType": "Bundle",
        "id": bundle_id,
        "meta": synthetic_meta(
            "transaction-writeback",
            "preliminary-evidence",
        ),
        "identifier": {
            "system": f"{IDENTIFIER_ROOT}/bundle",
            "value": bundle_id,
        },
        "type": "transaction",
        "timestamp": generated_utc,
        "entry": entries,
    }

print("✅ Deterministic FHIR R4 builders and Bundle helpers defined")
print(f"✅ Shared Device id: {DEVICE_ID}")
print("✅ All result builders preserve preliminary status")

✅ Deterministic FHIR R4 builders and Bundle helpers defined
✅ Shared Device id: nqc-monai-brats-segresnet-0-5-4
✅ All result builders preserve preliminary status


In [7]:
# Cell 4 — Build and persist linked FHIR evidence and self-contained transaction Bundles

def collect_references(
    value: Any,
    *,
    path: str = "$",
) -> list[dict[str, str]]:
    """
    Recursively collect every FHIR Reference.reference string.

    This helper is defined here because Cell 4 validates transaction-Bundle
    reference rewriting before Cell 5 performs the broader local validation.
    """
    collected: list[dict[str, str]] = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"

            if (
                key == "reference"
                and isinstance(child, str)
            ):
                collected.append(
                    {
                        "path": child_path,
                        "reference": child,
                    }
                )
            else:
                collected.extend(
                    collect_references(
                        child,
                        path=child_path,
                    )
                )

    elif isinstance(value, list):
        for index, child in enumerate(value):
            collected.extend(
                collect_references(
                    child,
                    path=f"{path}[{index}]",
                )
            )

    return collected


for stale_file in RESOURCE_ROOT.glob("*.json"):
    stale_file.unlink()
for stale_file in CASE_BUNDLE_ROOT.glob("*.json"):
    stale_file.unlink()

generated_utc = utc_now()
device_resource = build_device(generated_utc)

# Reload the five Notebook 01 source-context resources for each case:
# Patient, Condition, two ImagingStudy resources, and the prior reviewed Observation.
source_type_order = {
    "Patient": 0,
    "Condition": 1,
    "ImagingStudy": 2,
    "Observation": 3,
}
source_resources_by_case: dict[str, list[dict[str, Any]]] = {}

for case_id in CASE_ORDER:
    indexed_rows = [
        row
        for row in resource_index.get("resources", [])
        if row.get("case_id") == case_id
        and row.get("resource_type")
        in {"Patient", "Condition", "ImagingStudy", "Observation"}
    ]

    expected_source_counts = {
        "Patient": 1,
        "Condition": 1,
        "ImagingStudy": 2,
        "Observation": 1,
    }
    actual_source_counts = Counter(
        row["resource_type"] for row in indexed_rows
    )
    if dict(actual_source_counts) != expected_source_counts:
        raise AssertionError(
            f"{case_id}: unexpected source-context counts: "
            f"{dict(actual_source_counts)}"
        )

    loaded_source_resources = []
    for row in indexed_rows:
        path = PROJECT_ROOT / row["relative_path"]
        if not path.exists() or path.stat().st_size == 0:
            raise FileNotFoundError(path)
        resource = load_json(path)
        expected_reference = (
            f"{row['resource_type']}/{row['resource_id']}"
        )
        if resource_reference(resource) != expected_reference:
            raise AssertionError(
                f"{case_id}: source resource/index mismatch: "
                f"{expected_reference}"
            )
        loaded_source_resources.append(resource)

    loaded_source_resources.sort(
        key=lambda resource: (
            source_type_order[resource["resourceType"]],
            resource["id"],
        )
    )
    source_resources_by_case[case_id] = loaded_source_resources

case_resources: dict[str, dict[str, dict[str, Any]]] = {}
case_bundles: dict[str, dict[str, Any]] = {}
all_unique_resources: dict[str, dict[str, Any]] = {
    resource_reference(device_resource): device_resource
}

for case in sorted(
    cases,
    key=lambda row: CASE_ORDER.index(row["case_id"]),
):
    case_id = case["case_id"]

    if not bool(case["scenario_alignment_passed"]):
        raise AssertionError(
            f"{case_id}: Notebook 06 scenario alignment is false."
        )
    if case["workflow_state"]["ai_result_status"] != "preliminary":
        raise AssertionError(
            f"{case_id}: source AI result is not preliminary."
        )
    if bool(
        case["workflow_state"]["autonomous_finalization_allowed"]
    ):
        raise AssertionError(
            f"{case_id}: autonomous finalization was enabled."
        )

    observation = build_observation(case, generated_utc)
    report = build_diagnostic_report(
        case,
        observation,
        generated_utc,
    )
    task = build_task(case, observation, generated_utc)
    provenance = build_provenance(
        case,
        observation,
        report,
        task,
        generated_utc,
    )

    generated_resources = {
        "Observation": observation,
        "DiagnosticReport": report,
        "Task": task,
        "Provenance": provenance,
    }
    case_resources[case_id] = generated_resources

    for resource in generated_resources.values():
        all_unique_resources[resource_reference(resource)] = resource

    # Self-contained transaction: refresh all five synthetic source-context
    # resources, then write the shared Device and four newly generated resources.
    bundle_resources = [
        *source_resources_by_case[case_id],
        device_resource,
        observation,
        report,
        task,
        provenance,
    ]
    if len(bundle_resources) != 10:
        raise AssertionError(
            f"{case_id}: expected 10 self-contained transaction resources."
        )

    case_bundle = build_transaction_bundle(
        case_id,
        bundle_resources,
        generated_utc,
    )

    bundle_full_urls = {
        entry["fullUrl"]
        for entry in case_bundle["entry"]
    }
    bundle_resource_references = {
        resource_reference(resource)
        for resource in bundle_resources
    }

    unresolved_local_bundle_references = []
    for entry in case_bundle["entry"]:
        for item in collect_references(entry["resource"]):
            target = item["reference"]
            if target in bundle_resource_references:
                unresolved_local_bundle_references.append(
                    {
                        "source": resource_reference(
                            entry["resource"]
                        ),
                        "path": item["path"],
                        "target": target,
                    }
                )
            if (
                target.startswith("urn:uuid:")
                and target not in bundle_full_urls
            ):
                unresolved_local_bundle_references.append(
                    {
                        "source": resource_reference(
                            entry["resource"]
                        ),
                        "path": item["path"],
                        "target": target,
                    }
                )

    if unresolved_local_bundle_references:
        raise AssertionError(
            f"{case_id}: transaction reference rewriting failed: "
            + json.dumps(
                unresolved_local_bundle_references,
                indent=2,
            )
        )

    case_bundles[case_id] = case_bundle

    case_dir = OUTPUT_ROOT / case_id
    case_dir.mkdir(parents=True, exist_ok=True)

    for resource in bundle_resources:
        filename = (
            f"{resource['resourceType']}-{resource['id']}.json"
        )
        write_json(case_dir / filename, resource)

    write_json(
        CASE_BUNDLE_ROOT / f"{case_id}_transaction_bundle.json",
        case_bundle,
    )

for reference, resource in sorted(all_unique_resources.items()):
    write_json(
        RESOURCE_ROOT
        / f"{resource['resourceType']}-{resource['id']}.json",
        resource,
    )

master_collection_bundle = {
    "resourceType": "Bundle",
    "id": "nqc-notebook07-master-evidence-collection",
    "meta": synthetic_meta(
        "evidence-collection",
        "not-for-transaction",
    ),
    "identifier": {
        "system": f"{IDENTIFIER_ROOT}/bundle",
        "value": "nqc-notebook07-master-evidence-collection",
    },
    "type": "collection",
    "timestamp": generated_utc,
    "entry": [
        {
            "fullUrl": deterministic_full_url(
                resource["resourceType"],
                resource["id"],
            ),
            "resource": resource,
        }
        for resource in all_unique_resources.values()
    ],
}
write_json(MASTER_EVIDENCE_BUNDLE_PATH, master_collection_bundle)

resource_counts = Counter(
    resource["resourceType"]
    for resource in all_unique_resources.values()
)
expected_counts = {
    "Device": 1,
    "Observation": 3,
    "DiagnosticReport": 3,
    "Task": 3,
    "Provenance": 3,
}
if dict(resource_counts) != expected_counts:
    raise AssertionError(
        f"Unexpected generated FHIR evidence counts: "
        f"{dict(resource_counts)}"
    )

fhir_evidence_manifest = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "07",
    "generated_utc": generated_utc,
    "fhir_version": "4.0.1",
    "server_base_url": FHIR_BASE_URL,
    "data_governance": {
        "synthetic_fhir_only": True,
        "public_deidentified_imaging_only": True,
        "real_patient_data_allowed": False,
        "clinical_deployment_claimed": False,
    },
    "safety": {
        "observation_status": "preliminary",
        "diagnostic_report_status": "preliminary",
        "task_status": "requested",
        "human_review_executed": False,
        "autonomous_finalization_allowed": False,
    },
    "unique_resource_count": len(all_unique_resources),
    "resource_counts": dict(resource_counts),
    "transaction_bundle_count": len(case_bundles),
    "transaction_entry_count_per_case": 10,
    "source_context_entry_count_per_case": 5,
    "generated_evidence_entry_count_per_case": 5,
    "shared_device_reference": resource_reference(device_resource),
    "cases": [
        {
            "case_id": case_id,
            "patient_reference": next(
                case["patient_reference"]
                for case in cases
                if case["case_id"] == case_id
            ),
            "resources": {
                resource_type: resource_reference(resource)
                for resource_type, resource
                in case_resources[case_id].items()
            },
            "source_context_resources": [
                resource_reference(resource)
                for resource in source_resources_by_case[case_id]
            ],
            "transaction_bundle_file": (
                CASE_BUNDLE_ROOT
                / f"{case_id}_transaction_bundle.json"
            ).relative_to(PROJECT_ROOT).as_posix(),
        }
        for case_id in CASE_ORDER
    ],
    "resource_directory": RESOURCE_ROOT.relative_to(
        PROJECT_ROOT
    ).as_posix(),
    "master_collection_bundle_file": (
        MASTER_EVIDENCE_BUNDLE_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix()
    ),
}
write_json(
    FHIR_EVIDENCE_MANIFEST_PATH,
    fhir_evidence_manifest,
)

print("=" * 104)
print("✅ FHIR R4 evidence resources generated")
print(f"✅ Unique generated evidence resources: {len(all_unique_resources)}")
for resource_type, count in expected_counts.items():
    print(f" - {resource_type}: {count}")
print("✅ Three self-contained transaction Bundles: 10 entries each")
print("✅ Each transaction refreshes 5 synthetic source-context resources")
print(f"📁 Submission package: {OUTPUT_ROOT}")
print("=" * 104)

✅ FHIR R4 evidence resources generated
✅ Unique generated evidence resources: 13
 - Device: 1
 - Observation: 3
 - DiagnosticReport: 3
 - Task: 3
 - Provenance: 3
✅ Three self-contained transaction Bundles: 10 entries each
✅ Each transaction refreshes 5 synthetic source-context resources
📁 Submission package: /content/drive/MyDrive/neurofhir-qc/submission/fhir_resources/notebook_07


In [8]:
# Cell 5 — Run local FHIR structure, safety, and reference-integrity validation

REQUIRED_FIELDS = {
    "Device": {"resourceType", "id", "status", "deviceName"},
    "Observation": {
        "resourceType",
        "id",
        "status",
        "code",
        "subject",
        "effectiveDateTime",
        "valueQuantity",
    },
    "DiagnosticReport": {
        "resourceType",
        "id",
        "status",
        "code",
        "subject",
        "result",
    },
    "Task": {
        "resourceType",
        "id",
        "status",
        "intent",
        "focus",
        "for",
    },
    "Provenance": {
        "resourceType",
        "id",
        "target",
        "recorded",
        "agent",
    },
}

def collect_references(
    value: Any,
    *,
    path: str = "$",
) -> list[dict[str, str]]:
    collected: list[dict[str, str]] = []
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}"
            if (
                key == "reference"
                and isinstance(child, str)
            ):
                collected.append(
                    {
                        "path": child_path,
                        "reference": child,
                    }
                )
            else:
                collected.extend(
                    collect_references(child, path=child_path)
                )
    elif isinstance(value, list):
        for index, child in enumerate(value):
            collected.extend(
                collect_references(
                    child,
                    path=f"{path}[{index}]",
                )
            )
    return collected

def has_tag(
    resource: dict[str, Any],
    code: str,
) -> bool:
    return any(
        tag.get("system") == TAG_SYSTEM
        and tag.get("code") == code
        for tag in resource.get("meta", {}).get("tag", [])
    )

source_references = {
    f"{row['resource_type']}/{row['resource_id']}"
    for row in resource_index.get("resources", [])
}
generated_references = set(all_unique_resources)
resolvable_references = source_references | generated_references

resource_validation_rows: list[dict[str, Any]] = []
reference_rows: list[dict[str, Any]] = []
validation_errors: list[str] = []

for reference, resource in sorted(all_unique_resources.items()):
    resource_type = resource.get("resourceType")
    resource_id = resource.get("id")

    missing_fields = sorted(
        REQUIRED_FIELDS[resource_type].difference(resource)
    )
    if missing_fields:
        validation_errors.append(
            f"{reference}: missing {missing_fields}"
        )

    if not FHIR_ID_PATTERN.fullmatch(str(resource_id)):
        validation_errors.append(
            f"{reference}: invalid FHIR id"
        )

    if not has_tag(resource, "synthetic-demonstration"):
        validation_errors.append(
            f"{reference}: missing synthetic-data tag"
        )

    if resource_type == "Observation":
        if resource.get("status") != "preliminary":
            validation_errors.append(
                f"{reference}: Observation is not preliminary"
            )
        if resource.get("valueQuantity", {}).get("code") != "mL":
            validation_errors.append(
                f"{reference}: primary value is not UCUM mL"
            )
        if not resource.get("derivedFrom"):
            validation_errors.append(
                f"{reference}: missing source derivation"
            )

    if resource_type == "DiagnosticReport":
        if resource.get("status") != "preliminary":
            validation_errors.append(
                f"{reference}: DiagnosticReport is not preliminary"
            )
        if len(resource.get("result", [])) != 1:
            validation_errors.append(
                f"{reference}: expected one result Observation"
            )

    if resource_type == "Task":
        if resource.get("status") != "requested":
            validation_errors.append(
                f"{reference}: Task status is not requested"
            )
        if resource.get("intent") != "order":
            validation_errors.append(
                f"{reference}: Task intent is not order"
            )

    if resource_type == "Provenance":
        if len(resource.get("agent", [])) < 1:
            validation_errors.append(
                f"{reference}: missing algorithm agent"
            )
        if len(resource.get("target", [])) != 3:
            validation_errors.append(
                f"{reference}: expected three provenance targets"
            )

    resource_refs = collect_references(resource)
    for item in resource_refs:
        target = item["reference"]
        resolved = (
            target in resolvable_references
            or target.startswith("http://")
            or target.startswith("https://")
            or target.startswith("urn:")
        )
        reference_rows.append(
            {
                "source_reference": reference,
                "path": item["path"],
                "target_reference": target,
                "resolved": resolved,
            }
        )
        if not resolved:
            validation_errors.append(
                f"{reference}: unresolved {target} at {item['path']}"
            )

    resource_validation_rows.append(
        {
            "reference": reference,
            "resource_type": resource_type,
            "resource_id": resource_id,
            "missing_required_fields": missing_fields,
            "synthetic_tag_present": has_tag(
                resource,
                "synthetic-demonstration",
            ),
            "reference_count": len(resource_refs),
            "passed": not missing_fields,
        }
    )

for case_id, bundle in case_bundles.items():
    if bundle.get("type") != "transaction":
        validation_errors.append(
            f"{case_id}: Bundle is not a transaction"
        )
    if len(bundle.get("entry", [])) != 10:
        validation_errors.append(
            f"{case_id}: expected ten self-contained transaction entries"
        )
    for index, entry in enumerate(bundle.get("entry", [])):
        request = entry.get("request", {})
        resource = entry.get("resource")
        if request.get("method") != "PUT":
            validation_errors.append(
                f"{case_id}: entry {index} is not deterministic PUT"
            )
        expected_url = (
            f"{resource['resourceType']}/{resource['id']}"
            if isinstance(resource, dict)
            else None
        )
        if request.get("url") != expected_url:
            validation_errors.append(
                f"{case_id}: entry {index} request URL mismatch"
            )

for case_id in CASE_ORDER:
    task = case_resources[case_id]["Task"]
    observation = case_resources[case_id]["Observation"]
    report = case_resources[case_id]["DiagnosticReport"]
    provenance = case_resources[case_id]["Provenance"]

    if case_id == "low-confidence":
        if task.get("priority") != "urgent":
            validation_errors.append(
                "low-confidence: review Task is not urgent"
            )
        source_case = next(
            case for case in cases
            if case["case_id"] == case_id
        )
        if source_case["qc_category"] != "Manual review required":
            validation_errors.append(
                "low-confidence: source QC category changed"
            )

    if task["focus"]["reference"] != resource_reference(observation):
        validation_errors.append(
            f"{case_id}: Task focus mismatch"
        )
    if report["result"][0]["reference"] != resource_reference(
        observation
    ):
        validation_errors.append(
            f"{case_id}: report result mismatch"
        )

    provenance_targets = {
        item["reference"]
        for item in provenance["target"]
    }
    required_targets = {
        resource_reference(observation),
        resource_reference(report),
        resource_reference(task),
    }
    if provenance_targets != required_targets:
        validation_errors.append(
            f"{case_id}: provenance targets mismatch"
        )

if validation_errors:
    raise AssertionError(
        "Local FHIR validation failed:\n"
        + "\n".join(f" - {item}" for item in validation_errors)
    )

reference_integrity_rate = (
    sum(bool(row["resolved"]) for row in reference_rows)
    / len(reference_rows)
    if reference_rows
    else 1.0
)
local_validation_report = {
    "project_name": project_config["project_name"],
    "validated_utc": utc_now(),
    "unique_resource_count": len(all_unique_resources),
    "resource_validation_pass_rate": 1.0,
    "reference_count": len(reference_rows),
    "reference_integrity_rate": round(
        reference_integrity_rate,
        6,
    ),
    "preliminary_observation_rate": (
        sum(
            resource.get("status") == "preliminary"
            for resource in all_unique_resources.values()
            if resource["resourceType"] == "Observation"
        )
        / 3
    ),
    "preliminary_report_rate": (
        sum(
            resource.get("status") == "preliminary"
            for resource in all_unique_resources.values()
            if resource["resourceType"] == "DiagnosticReport"
        )
        / 3
    ),
    "review_task_requested_rate": (
        sum(
            resource.get("status") == "requested"
            for resource in all_unique_resources.values()
            if resource["resourceType"] == "Task"
        )
        / 3
    ),
    "autonomous_finalization_allowed": False,
    "rows": resource_validation_rows,
    "references": reference_rows,
}
write_json(LOCAL_VALIDATION_PATH, local_validation_report)

print("=" * 104)
print("✅ Local FHIR structure validation passed: 13/13 resources")
print(
    f"✅ Reference integrity: "
    f"{sum(row['resolved'] for row in reference_rows)}/"
    f"{len(reference_rows)} "
    f"({reference_integrity_rate:.1%})"
)
print("✅ 3/3 Observations and DiagnosticReports remain preliminary")
print("✅ 3/3 human-review Tasks remain requested")
print("=" * 104)

✅ Local FHIR structure validation passed: 13/13 resources
✅ Reference integrity: 51/51 (100.0%)
✅ 3/3 Observations and DiagnosticReports remain preliminary
✅ 3/3 human-review Tasks remain requested


In [9]:
# Cell 6 — Pre-write validation of the three self-contained transaction Bundles

for stale_file in VALIDATION_ROOT.glob("*_operation_outcome.json"):
    stale_file.unlink()

def validation_passed(
    response: requests.Response,
    payload: Any,
) -> tuple[bool, list[dict[str, Any]]]:
    issues = operation_outcome_summary(payload)
    blocking = [
        issue
        for issue in issues
        if issue.get("severity") in {"fatal", "error"}
    ]
    return response.ok and not blocking, issues

def format_blocking_issues(
    label: str,
    issues: list[dict[str, Any]],
) -> str:
    blocking = [
        issue
        for issue in issues
        if issue.get("severity") in {"fatal", "error"}
    ]
    if not blocking:
        return f"{label}: no blocking issues"
    lines = [f"{label}:"]
    for issue in blocking:
        expression = issue.get("expression") or []
        location = issue.get("location") or []
        lines.append(
            "  - "
            + str(
                issue.get("diagnostics")
                or issue.get("details")
                or issue.get("code")
            )
            + (
                f" | expression={expression}"
                if expression
                else ""
            )
            + (
                f" | location={location}"
                if location
                else ""
            )
        )
    return "\n".join(lines)

prewrite_bundle_validation_rows: list[dict[str, Any]] = []
prewrite_validation_issues: dict[str, list[dict[str, Any]]] = {}

for index, case_id in enumerate(CASE_ORDER, start=1):
    bundle = case_bundles[case_id]
    label = f"Bundle/{bundle['id']}"

    response = request_fhir(
        "POST",
        "Bundle/$validate",
        payload=bundle,
        purpose=f"Pre-write FHIR Bundle $validate: {case_id}",
        retry_read_like=True,
    )
    try:
        response_payload = response.json()
    except Exception:
        response_payload = {
            "resourceType": "OperationOutcome",
            "issue": [
                {
                    "severity": "error",
                    "code": "exception",
                    "diagnostics": response.text[:1000],
                }
            ],
        }

    passed, issues = validation_passed(
        response,
        response_payload,
    )
    prewrite_validation_issues[label] = issues

    outcome_path = (
        VALIDATION_ROOT
        / f"{index:02d}_prewrite_{case_id}_bundle_operation_outcome.json"
    )
    write_json(outcome_path, response_payload)

    prewrite_bundle_validation_rows.append(
        {
            "phase": "prewrite-bundle",
            "case_id": case_id,
            "label": label,
            "endpoint": "Bundle/$validate",
            "status_code": response.status_code,
            "passed": passed,
            "blocking_issue_count": sum(
                issue.get("severity") in {"fatal", "error"}
                for issue in issues
            ),
            "warning_issue_count": sum(
                issue.get("severity") == "warning"
                for issue in issues
            ),
            "information_issue_count": sum(
                issue.get("severity") == "information"
                for issue in issues
            ),
            "operation_outcome_file": outcome_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

failed_prewrite = [
    row
    for row in prewrite_bundle_validation_rows
    if not row["passed"]
]
if failed_prewrite:
    details = []
    for row in failed_prewrite:
        details.append(
            format_blocking_issues(
                row["label"],
                prewrite_validation_issues[row["label"]],
            )
        )
    raise AssertionError(
        "Pre-write self-contained Bundle validation failed:\n"
        + "\n".join(details)
    )

print("=" * 104)
print("✅ Pre-write FHIR validation passed: 3/3 self-contained Bundles")
print("✅ Every validated Bundle contains 10 deterministic PUT entries")
print("✅ Generated-resource references resolve inside each Bundle")
print("➡️ Proceeding to transaction write-back")
print("=" * 104)

✅ Pre-write FHIR validation passed: 3/3 self-contained Bundles
✅ Every validated Bundle contains 10 deterministic PUT entries
✅ Generated-resource references resolve inside each Bundle
➡️ Proceeding to transaction write-back


In [10]:
# Cell 7 — Write self-contained transactions, read back generated evidence, and validate post-write

if not ALLOW_SYNTHETIC_FHIR_WRITEBACK:
    raise RuntimeError(
        "Synthetic FHIR write-back is disabled. Review the data boundary "
        "and explicitly set ALLOW_SYNTHETIC_FHIR_WRITEBACK = True."
    )

transaction_rows: list[dict[str, Any]] = []
transaction_entry_rows: list[dict[str, Any]] = []

for stale_file in SERVER_RESPONSE_ROOT.glob("*.json"):
    stale_file.unlink()
for stale_file in READBACK_ROOT.glob("*.json"):
    stale_file.unlink()

for case_id in CASE_ORDER:
    bundle = case_bundles[case_id]

    # Transactions are deliberately not automatically retried.
    # Deterministic PUT entries make a manual rerun idempotent.
    response = request_fhir(
        "POST",
        "",
        payload=bundle,
        purpose=f"Write self-contained synthetic evidence transaction: {case_id}",
        retry_read_like=False,
    )
    try:
        response_payload = response.json()
    except Exception as exc:
        raise RuntimeError(
            f"{case_id}: transaction did not return FHIR JSON: "
            f"{response.text[:1000]}"
        ) from exc

    response_path = (
        SERVER_RESPONSE_ROOT
        / f"{case_id}_transaction_response.json"
    )
    write_json(response_path, response_payload)

    if response_payload.get("resourceType") != "Bundle":
        raise AssertionError(
            f"{case_id}: server did not return a Bundle."
        )
    if response_payload.get("type") != "transaction-response":
        raise AssertionError(
            f"{case_id}: response is not transaction-response."
        )

    response_entries = response_payload.get("entry", [])
    expected_entry_count = len(bundle.get("entry", []))
    if expected_entry_count != 10:
        raise AssertionError(
            f"{case_id}: expected a 10-entry transaction."
        )
    if len(response_entries) != expected_entry_count:
        raise AssertionError(
            f"{case_id}: transaction-response entry count mismatch."
        )

    successful_entries = 0
    for index, response_entry in enumerate(response_entries):
        response_details = response_entry.get("response", {})
        status_text = str(response_details.get("status", ""))
        status_match = re.match(r"^(\d{3})", status_text)
        status_code = (
            int(status_match.group(1))
            if status_match
            else 0
        )
        passed = 200 <= status_code < 300
        successful_entries += int(passed)

        transaction_entry_rows.append(
            {
                "case_id": case_id,
                "entry_index": index,
                "request_url": bundle["entry"][index][
                    "request"
                ]["url"],
                "response_status": status_text,
                "response_location": response_details.get(
                    "location"
                ),
                "etag": response_details.get("etag"),
                "last_modified": response_details.get(
                    "lastModified"
                ),
                "passed": passed,
            }
        )

    transaction_passed = (
        response.ok
        and successful_entries == expected_entry_count
    )
    transaction_rows.append(
        {
            "case_id": case_id,
            "http_status_code": response.status_code,
            "entry_count": expected_entry_count,
            "successful_entry_count": successful_entries,
            "transaction_passed": transaction_passed,
            "response_file": response_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

failed_transactions = [
    row
    for row in transaction_rows
    if not row["transaction_passed"]
]
if failed_transactions:
    raise AssertionError(
        "FHIR transaction write-back failed:\n"
        + "\n".join(
            f" - {row['case_id']}: "
            f"{row['successful_entry_count']}/"
            f"{row['entry_count']} entries"
            for row in failed_transactions
        )
    )

write_json(
    TRANSACTION_REPORT_PATH,
    {
        "project_name": project_config["project_name"],
        "written_utc": utc_now(),
        "server_base_url": FHIR_BASE_URL,
        "transaction_count": len(transaction_rows),
        "transaction_success_count": sum(
            row["transaction_passed"]
            for row in transaction_rows
        ),
        "transaction_success_rate": round(
            sum(
                row["transaction_passed"]
                for row in transaction_rows
            )
            / len(transaction_rows),
            6,
        ),
        "submitted_entry_count": sum(
            row["entry_count"]
            for row in transaction_rows
        ),
        "successful_entry_count": sum(
            row["successful_entry_count"]
            for row in transaction_rows
        ),
        "source_context_entry_count": 15,
        "generated_evidence_entry_count_including_repeated_device": 15,
        "transactions": transaction_rows,
        "entries": transaction_entry_rows,
    },
)

def critical_signature(
    resource: dict[str, Any],
) -> dict[str, Any]:
    resource_type = resource["resourceType"]
    signature: dict[str, Any] = {
        "resourceType": resource_type,
        "id": resource["id"],
    }

    if resource_type == "Device":
        signature.update(
            {
                "status": resource.get("status"),
                "identifier": resource.get("identifier"),
                "deviceName": resource.get("deviceName"),
                "version": resource.get("version"),
                "modelNumber": resource.get("modelNumber"),
            }
        )
    elif resource_type == "Observation":
        signature.update(
            {
                "status": resource.get("status"),
                "subject": resource.get("subject"),
                "focus": resource.get("focus"),
                "device": resource.get("device"),
                "effectiveDateTime": resource.get(
                    "effectiveDateTime"
                ),
                "valueQuantity": resource.get("valueQuantity"),
                "derivedFrom": resource.get("derivedFrom"),
                "component": resource.get("component"),
            }
        )
    elif resource_type == "DiagnosticReport":
        signature.update(
            {
                "status": resource.get("status"),
                "subject": resource.get("subject"),
                "imagingStudy": resource.get("imagingStudy"),
                "result": resource.get("result"),
                "conclusion": resource.get("conclusion"),
                "conclusionCode": resource.get("conclusionCode"),
            }
        )
    elif resource_type == "Task":
        signature.update(
            {
                "status": resource.get("status"),
                "intent": resource.get("intent"),
                "priority": resource.get("priority"),
                "focus": resource.get("focus"),
                "for": resource.get("for"),
                "requester": resource.get("requester"),
                "input": resource.get("input"),
            }
        )
    elif resource_type == "Provenance":
        signature.update(
            {
                "target": resource.get("target"),
                "recorded": resource.get("recorded"),
                "activity": resource.get("activity"),
                "agent": resource.get("agent"),
                "entity": resource.get("entity"),
            }
        )
    return signature

readback_rows: list[dict[str, Any]] = []

for reference, local_resource in sorted(
    all_unique_resources.items()
):
    response = request_fhir(
        "GET",
        reference,
        purpose=f"Direct read-back: {reference}",
        retry_read_like=True,
    )
    if not response.ok:
        raise RuntimeError(
            f"Direct read failed for {reference}: "
            f"{response.status_code} {response.text[:500]}"
        )

    server_resource = response.json()
    if server_resource.get("resourceType") != local_resource[
        "resourceType"
    ]:
        raise AssertionError(
            f"{reference}: server resource type changed."
        )
    if server_resource.get("id") != local_resource["id"]:
        raise AssertionError(
            f"{reference}: server resource id changed."
        )

    local_signature = critical_signature(local_resource)
    server_signature = critical_signature(server_resource)
    signature_preserved = local_signature == server_signature

    readback_path = (
        READBACK_ROOT
        / f"{local_resource['resourceType']}-"
          f"{local_resource['id']}.json"
    )
    write_json(readback_path, server_resource)

    readback_rows.append(
        {
            "reference": reference,
            "http_status_code": response.status_code,
            "critical_signature_preserved": (
                signature_preserved
            ),
            "readback_file": readback_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

failed_readbacks = [
    row
    for row in readback_rows
    if not row["critical_signature_preserved"]
]
if failed_readbacks:
    raise AssertionError(
        "FHIR read-back critical-field preservation failed:\n"
        + "\n".join(
            f" - {row['reference']}"
            for row in failed_readbacks
        )
    )

readback_success_rate = (
    sum(
        row["critical_signature_preserved"]
        for row in readback_rows
    )
    / len(readback_rows)
)

write_json(
    READBACK_REPORT_PATH,
    {
        "project_name": project_config["project_name"],
        "read_utc": utc_now(),
        "server_base_url": FHIR_BASE_URL,
        "unique_resource_count": len(readback_rows),
        "direct_read_success_rate": 1.0,
        "critical_field_preservation_rate": round(
            readback_success_rate,
            6,
        ),
        "rows": readback_rows,
    },
)

# Validate generated resources only after transaction write-back, so every
# linked Device, Observation, DiagnosticReport, Task, Patient, Condition,
# ImagingStudy, and prior Observation reference is available on the server.
postwrite_resource_validation_rows: list[dict[str, Any]] = []
postwrite_validation_issues: dict[str, list[dict[str, Any]]] = {}

for offset, (
    reference,
    resource,
) in enumerate(
    sorted(all_unique_resources.items()),
    start=4,
):
    endpoint = f"{resource['resourceType']}/$validate"
    response = request_fhir(
        "POST",
        endpoint,
        payload=resource,
        purpose=f"Post-write FHIR resource $validate: {reference}",
        retry_read_like=True,
    )
    try:
        response_payload = response.json()
    except Exception:
        response_payload = {
            "resourceType": "OperationOutcome",
            "issue": [
                {
                    "severity": "error",
                    "code": "exception",
                    "diagnostics": response.text[:1000],
                }
            ],
        }

    passed, issues = validation_passed(
        response,
        response_payload,
    )
    postwrite_validation_issues[reference] = issues

    safe_label = re.sub(
        r"[^A-Za-z0-9\-\.]+",
        "_",
        reference,
    )
    outcome_path = (
        VALIDATION_ROOT
        / f"{offset:02d}_postwrite_{safe_label}_operation_outcome.json"
    )
    write_json(outcome_path, response_payload)

    postwrite_resource_validation_rows.append(
        {
            "phase": "postwrite-resource",
            "label": reference,
            "endpoint": endpoint,
            "status_code": response.status_code,
            "passed": passed,
            "blocking_issue_count": sum(
                issue.get("severity") in {"fatal", "error"}
                for issue in issues
            ),
            "warning_issue_count": sum(
                issue.get("severity") == "warning"
                for issue in issues
            ),
            "information_issue_count": sum(
                issue.get("severity") == "information"
                for issue in issues
            ),
            "operation_outcome_file": outcome_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        }
    )

failed_postwrite = [
    row
    for row in postwrite_resource_validation_rows
    if not row["passed"]
]
if failed_postwrite:
    details = []
    for row in failed_postwrite:
        details.append(
            format_blocking_issues(
                row["label"],
                postwrite_validation_issues[row["label"]],
            )
        )
    raise AssertionError(
        "Post-write generated-resource validation failed:\n"
        + "\n".join(details)
    )

server_validation_rows = [
    *prewrite_bundle_validation_rows,
    *postwrite_resource_validation_rows,
]
server_validation_pass_rate = (
    sum(bool(row["passed"]) for row in server_validation_rows)
    / len(server_validation_rows)
)

write_json(
    SERVER_VALIDATION_PATH,
    {
        "project_name": project_config["project_name"],
        "validated_utc": utc_now(),
        "server_base_url": FHIR_BASE_URL,
        "fhir_version": fhir_version,
        "validation_strategy": (
            "Validate three self-contained transaction Bundles before "
            "write-back, then validate 13 generated resources after "
            "their references exist on the server."
        ),
        "validation_target_count": len(
            server_validation_rows
        ),
        "resource_validation_target_count": 13,
        "bundle_validation_target_count": 3,
        "validation_pass_rate": round(
            server_validation_pass_rate,
            6,
        ),
        "rows": server_validation_rows,
    },
)

write_json(NETWORK_LOG_PATH, network_log)

print("=" * 104)
print("✅ Self-contained FHIR transaction write-back succeeded: 3/3 Bundles")
print(
    f"✅ Transaction entries succeeded: "
    f"{sum(row['successful_entry_count'] for row in transaction_rows)}/"
    f"{sum(row['entry_count'] for row in transaction_rows)}"
)
print("✅ Direct read-back succeeded: 13/13 generated resources")
print("✅ Critical-field preservation: 100.0%")
print("✅ Post-write FHIR validation passed: 13/13 generated resources")
print("✅ Total server validation targets passed: 16/16")
print("=" * 104)

✅ Self-contained FHIR transaction write-back succeeded: 3/3 Bundles
✅ Transaction entries succeeded: 30/30
✅ Direct read-back succeeded: 13/13 generated resources
✅ Critical-field preservation: 100.0%
✅ Post-write FHIR validation passed: 13/13 generated resources
✅ Total server validation targets passed: 16/16


In [11]:
# Cell 8 — Build the FHIR reference graph and verify the server-returned evidence graph

graph_edges: list[dict[str, Any]] = []

for source_reference, local_resource in sorted(
    all_unique_resources.items()
):
    for item in collect_references(local_resource):
        target_reference = item["reference"]
        graph_edges.append(
            {
                "source_reference": source_reference,
                "source_type": local_resource["resourceType"],
                "path": item["path"],
                "target_reference": target_reference,
                "target_type": (
                    target_reference.split("/", 1)[0]
                    if "/" in target_reference
                    else None
                ),
                "target_is_generated": (
                    target_reference in generated_references
                ),
                "target_is_source_context": (
                    target_reference in source_references
                ),
                "resolved": (
                    target_reference in resolvable_references
                    or target_reference.startswith("http://")
                    or target_reference.startswith("https://")
                    or target_reference.startswith("urn:")
                ),
            }
        )

unresolved_edges = [
    edge for edge in graph_edges
    if not edge["resolved"]
]
if unresolved_edges:
    raise AssertionError(
        "FHIR graph contains unresolved references:\n"
        + "\n".join(
            f" - {edge['source_reference']} -> "
            f"{edge['target_reference']}"
            for edge in unresolved_edges
        )
    )

case_graphs = []
for case_id in CASE_ORDER:
    resources = case_resources[case_id]
    case_references = {
        resource_reference(resource)
        for resource in resources.values()
    }
    case_references.add(f"Device/{DEVICE_ID}")

    case_edges = [
        edge
        for edge in graph_edges
        if edge["source_reference"] in case_references
    ]
    case_graphs.append(
        {
            "case_id": case_id,
            "generated_resource_references": sorted(
                case_references
            ),
            "edge_count": len(case_edges),
            "all_references_resolved": all(
                edge["resolved"]
                for edge in case_edges
            ),
            "edges": case_edges,
        }
    )

reference_graph = {
    "project_name": project_config["project_name"],
    "generated_utc": utc_now(),
    "node_count": len(
        set(generated_references) | source_references
    ),
    "generated_node_count": len(generated_references),
    "source_context_node_count": len(source_references),
    "edge_count": len(graph_edges),
    "reference_integrity_rate": 1.0,
    "cases": case_graphs,
    "edges": graph_edges,
}
write_json(REFERENCE_GRAPH_JSON, reference_graph)

with REFERENCE_GRAPH_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "source_reference",
            "source_type",
            "path",
            "target_reference",
            "target_type",
            "target_is_generated",
            "target_is_source_context",
            "resolved",
        ],
    )
    writer.writeheader()
    writer.writerows(graph_edges)

for case_graph in case_graphs:
    if not case_graph["all_references_resolved"]:
        raise AssertionError(
            f"{case_graph['case_id']}: graph resolution failed."
        )

print("=" * 104)
print(f"✅ FHIR reference graph created: {len(graph_edges)} edges")
print("✅ Generated and source-context references resolve")
print("✅ Three case-level evidence graphs archived")
print(f"📄 Graph JSON: {REFERENCE_GRAPH_JSON}")
print("=" * 104)

✅ FHIR reference graph created: 51 edges
✅ Generated and source-context references resolve
✅ Three case-level evidence graphs archived
📄 Graph JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_07_fhir_evidence/fhir_reference_graph.json


In [12]:
# Cell 9 — Create reusable FHIR builders, write-back service, verifier, requirements, and documentation

BUILDERS_PATH = (
    PROJECT_ROOT
    / "backend/app/fhir_builders/evidence_builders.py"
)
WRITEBACK_SERVICE_PATH = (
    PROJECT_ROOT
    / "backend/app/services/fhir_writeback_service.py"
)
VERIFY_SCRIPT_PATH = (
    PROJECT_ROOT / "scripts/verify_fhir_evidence.py"
)
WRITEBACK_SCRIPT_PATH = (
    PROJECT_ROOT / "scripts/writeback_fhir_evidence.py"
)
REQUIREMENTS_PATH = (
    PROJECT_ROOT / "requirements/fhir_evidence.txt"
)
TECH_DOC_PATH = (
    DOC_ROOT / "FHIR_EVIDENCE_AND_WRITEBACK.md"
)

builders_source = r"""
from __future__ import annotations

import math
import re
import uuid
from typing import Any, Iterable

PROJECT_SYSTEM_ROOT = "https://neurofhir-qc.org/fhir"
IDENTIFIER_ROOT = f"{PROJECT_SYSTEM_ROOT}/identifier"
CODE_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/neurofhir-qc"
TAG_SYSTEM = f"{PROJECT_SYSTEM_ROOT}/CodeSystem/data-origin"
FHIR_ID_PATTERN = re.compile(r"^[A-Za-z0-9\-\.]{1,64}$")

def slug(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9\-\.]+", "-", value.strip())
    cleaned = re.sub(r"-+", "-", cleaned).strip("-")[:64].rstrip("-")
    if not cleaned or not FHIR_ID_PATTERN.fullmatch(cleaned):
        raise ValueError(f"Invalid FHIR id: {cleaned!r}")
    return cleaned

def coding(system: str, code: str, display: str) -> dict[str, str]:
    return {"system": system, "code": code, "display": display}

def concept(system: str, code: str, display: str) -> dict[str, Any]:
    return {
        "coding": [coding(system, code, display)],
        "text": display,
    }

def quantity(value: float, unit: str, code: str) -> dict[str, Any]:
    if not math.isfinite(float(value)):
        raise ValueError("Quantity must be finite")
    return {
        "value": round(float(value), 6),
        "unit": unit,
        "system": "http://unitsofmeasure.org",
        "code": code,
    }

def resource_reference(resource: dict[str, Any]) -> str:
    return f"{resource['resourceType']}/{resource['id']}"

def transaction_entry(resource: dict[str, Any]) -> dict[str, Any]:
    namespace = uuid.UUID("3a179bae-8ee9-4f7f-9a81-faf1c77439fc")
    reference = resource_reference(resource)
    return {
        "fullUrl": f"urn:uuid:{uuid.uuid5(namespace, reference)}",
        "resource": resource,
        "request": {
            "method": "PUT",
            "url": reference,
        },
    }

def transaction_bundle(
    bundle_id: str,
    timestamp: str,
    resources: Iterable[dict[str, Any]],
) -> dict[str, Any]:
    bundle_id = slug(bundle_id)
    return {
        "resourceType": "Bundle",
        "id": bundle_id,
        "type": "transaction",
        "timestamp": timestamp,
        "entry": [transaction_entry(resource) for resource in resources],
    }
"""

writeback_service_source = r"""
from __future__ import annotations

import os
from typing import Any

import requests

class FHIRWritebackError(RuntimeError):
    pass

class FHIRWritebackService:
    def __init__(
        self,
        base_url: str,
        timeout_seconds: int = 60,
    ) -> None:
        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds
        self.session = requests.Session()
        self.session.headers.update(
            {
                "Accept": "application/fhir+json, application/json",
                "Content-Type": "application/fhir+json",
                "User-Agent": "NeuroFHIR-QC-FHIRWritebackService",
            }
        )

    def transaction(
        self,
        bundle: dict[str, Any],
        *,
        allow_synthetic_writeback: bool,
    ) -> dict[str, Any]:
        if not allow_synthetic_writeback:
            raise FHIRWritebackError(
                "Write-back is disabled. An explicit synthetic-only "
                "authorization is required."
            )
        if bundle.get("resourceType") != "Bundle":
            raise FHIRWritebackError("Payload is not a Bundle")
        if bundle.get("type") != "transaction":
            raise FHIRWritebackError("Bundle is not a transaction")
        if not bundle.get("entry"):
            raise FHIRWritebackError("Transaction Bundle has no entries")

        response = self.session.post(
            self.base_url,
            json=bundle,
            timeout=self.timeout_seconds,
        )
        if not response.ok:
            raise FHIRWritebackError(
                f"FHIR transaction failed: {response.status_code} "
                f"{response.text[:1000]}"
            )
        payload = response.json()
        if payload.get("resourceType") != "Bundle":
            raise FHIRWritebackError(
                "FHIR transaction did not return a Bundle"
            )
        if payload.get("type") != "transaction-response":
            raise FHIRWritebackError(
                "FHIR response is not transaction-response"
            )
        return payload

def explicit_environment_authorization() -> bool:
    return (
        os.getenv("ALLOW_SYNTHETIC_FHIR_WRITEBACK", "false")
        .strip()
        .lower()
        == "true"
    )
"""

verify_script_source = r"""
from __future__ import annotations

import json
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]
MANIFEST = (
    PROJECT_ROOT
    / "submission/fhir_resources/notebook_07/fhir_evidence_manifest.json"
)
LOCAL = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/local_validation_report.json"
)
SERVER = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/server_validation_report.json"
)
TRANSACTION = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/transaction_writeback_report.json"
)
READBACK = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/readback_integrity_report.json"
)
GRAPH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence/fhir_reference_graph.json"
)

def load(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def main() -> None:
    manifest = load(MANIFEST)
    local = load(LOCAL)
    server = load(SERVER)
    transaction = load(TRANSACTION)
    readback = load(READBACK)
    graph = load(GRAPH)

    if int(manifest.get("unique_resource_count", 0)) != 13:
        raise SystemExit("Expected 13 unique FHIR resources")
    if int(manifest.get("transaction_bundle_count", 0)) != 3:
        raise SystemExit("Expected three transaction Bundles")
    if float(local.get("resource_validation_pass_rate", 0)) != 1.0:
        raise SystemExit("Local FHIR validation did not pass")
    if float(local.get("reference_integrity_rate", 0)) != 1.0:
        raise SystemExit("Local reference integrity did not pass")
    if float(server.get("validation_pass_rate", 0)) != 1.0:
        raise SystemExit("Server FHIR validation did not pass")
    if float(transaction.get("transaction_success_rate", 0)) != 1.0:
        raise SystemExit("FHIR transaction write-back did not pass")
    if float(readback.get("direct_read_success_rate", 0)) != 1.0:
        raise SystemExit("Direct read-back did not pass")
    if float(readback.get("critical_field_preservation_rate", 0)) != 1.0:
        raise SystemExit("Critical-field preservation did not pass")
    if float(graph.get("reference_integrity_rate", 0)) != 1.0:
        raise SystemExit("FHIR graph references did not resolve")

    print("Notebook 07 FHIR evidence passed persisted-artifact verification.")

if __name__ == "__main__":
    main()
"""

writeback_script_source = r"""
from __future__ import annotations

import json
from pathlib import Path

from backend.app.services.fhir_writeback_service import (
    FHIRWritebackService,
    explicit_environment_authorization,
)

PROJECT_ROOT = Path(__file__).resolve().parents[1]
BUNDLE_ROOT = (
    PROJECT_ROOT
    / "submission/fhir_resources/notebook_07/case_bundles"
)
SERVER_URL = "https://hapi.fhir.org/baseR4"

def main() -> None:
    authorized = explicit_environment_authorization()
    service = FHIRWritebackService(SERVER_URL)

    for bundle_path in sorted(BUNDLE_ROOT.glob("*_transaction_bundle.json")):
        with bundle_path.open("r", encoding="utf-8") as handle:
            bundle = json.load(handle)
        response = service.transaction(
            bundle,
            allow_synthetic_writeback=authorized,
        )
        print(
            bundle_path.name,
            response.get("type"),
            len(response.get("entry", [])),
        )

if __name__ == "__main__":
    main()
"""

technical_documentation = f"""
# NeuroFHIR-QC FHIR Evidence and Transaction Write-back

Generated by `{NOTEBOOK_FILENAME}`.

## FHIR R4 resources

Notebook 07 creates:

- one shared `Device` identifying the pinned model/software;
- three preliminary AI-derived tumor-volume `Observation` resources;
- three preliminary `DiagnosticReport` resources;
- three requested human-review `Task` resources;
- three algorithmic-generation `Provenance` resources;
- three deterministic transaction Bundles.

## Resource-state boundary

Notebook 07 does not execute human review.

- Observation: `preliminary`
- DiagnosticReport: `preliminary`
- Task: `requested`
- Provenance: algorithmic generation only
- no human reviewer identity
- no accepted, rejected, correction-required or final state

Notebook 08 owns the review transition.

## Interoperability evidence

The notebook archives:

- CapabilityStatement;
- local structure and reference validation;
- server `$validate` OperationOutcome resources;
- three transaction-response Bundles;
- direct read-back copies;
- critical-field comparison;
- FHIR reference graph;
- network timing log;
- SHA-256 inventory.

## Write-back safety

The reusable application service keeps write-back disabled unless the environment variable
`ALLOW_SYNTHETIC_FHIR_WRITEBACK=true` is set. Only synthetic FHIR records and public
de-identified research imaging context are permitted.

## Interpretation boundary

The workflow is a research demonstration. Engineering QC and longitudinal categories are not
clinical probabilities, validated response criteria, diagnostic conclusions or safety guarantees.
"""

for path in (
    BUILDERS_PATH,
    WRITEBACK_SERVICE_PATH,
    VERIFY_SCRIPT_PATH,
    WRITEBACK_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    TECH_DOC_PATH,
):
    path.parent.mkdir(parents=True, exist_ok=True)

BUILDERS_PATH.write_text(
    textwrap.dedent(builders_source).strip() + "\n",
    encoding="utf-8",
)
WRITEBACK_SERVICE_PATH.write_text(
    textwrap.dedent(writeback_service_source).strip() + "\n",
    encoding="utf-8",
)
VERIFY_SCRIPT_PATH.write_text(
    textwrap.dedent(verify_script_source).strip() + "\n",
    encoding="utf-8",
)
WRITEBACK_SCRIPT_PATH.write_text(
    textwrap.dedent(writeback_script_source).strip() + "\n",
    encoding="utf-8",
)
REQUIREMENTS_PATH.write_text(
    "\n".join(
        [
            f"requests=={runtime_versions['requests']}",
            f"urllib3=={runtime_versions['urllib3']}",
        ]
    )
    + "\n",
    encoding="utf-8",
)
TECH_DOC_PATH.write_text(
    textwrap.dedent(technical_documentation).strip() + "\n",
    encoding="utf-8",
)

for path in (
    BUILDERS_PATH,
    WRITEBACK_SERVICE_PATH,
    VERIFY_SCRIPT_PATH,
    WRITEBACK_SCRIPT_PATH,
):
    compile(
        path.read_text(encoding="utf-8"),
        str(path),
        "exec",
    )

subprocess.check_call(
    [sys.executable, str(VERIFY_SCRIPT_PATH)]
)

print("=" * 104)
print("✅ Reusable FHIR builders and write-back service created")
print("✅ Verification and explicit-writeback scripts created")
print("✅ Generated Python files passed syntax compilation")
print(f"📘 Documentation: {TECH_DOC_PATH}")
print("=" * 104)

✅ Reusable FHIR builders and write-back service created
✅ Verification and explicit-writeback scripts created
✅ Generated Python files passed syntax compilation
📘 Documentation: /content/drive/MyDrive/neurofhir-qc/docs/FHIR_EVIDENCE_AND_WRITEBACK.md


In [13]:
# Cell 10 — Final audit, checksum inventory, and Notebook 08 gate

write_json(NETWORK_LOG_PATH, network_log)

required_artifacts = [
    FHIR_EVIDENCE_MANIFEST_PATH,
    MASTER_EVIDENCE_BUNDLE_PATH,
    LOCAL_VALIDATION_PATH,
    SERVER_VALIDATION_PATH,
    TRANSACTION_REPORT_PATH,
    READBACK_REPORT_PATH,
    REFERENCE_GRAPH_JSON,
    REFERENCE_GRAPH_CSV,
    NETWORK_LOG_PATH,
    BUILDERS_PATH,
    WRITEBACK_SERVICE_PATH,
    VERIFY_SCRIPT_PATH,
    WRITEBACK_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    TECH_DOC_PATH,
]
required_artifacts += sorted(RESOURCE_ROOT.glob("*.json"))
required_artifacts += sorted(CASE_BUNDLE_ROOT.glob("*.json"))
required_artifacts += sorted(
    SERVER_RESPONSE_ROOT.glob("*.json")
)
required_artifacts += sorted(READBACK_ROOT.glob("*.json"))
required_artifacts += sorted(
    VALIDATION_ROOT.glob("*_operation_outcome.json")
)

missing_artifacts = [
    str(path)
    for path in required_artifacts
    if not path.exists() or path.stat().st_size == 0
]
if missing_artifacts:
    raise FileNotFoundError(
        "Notebook 07 evidence is incomplete:\n"
        + "\n".join(f" - {path}" for path in missing_artifacts)
    )

persisted_manifest = load_json(FHIR_EVIDENCE_MANIFEST_PATH)
persisted_local = load_json(LOCAL_VALIDATION_PATH)
persisted_server = load_json(SERVER_VALIDATION_PATH)
persisted_transaction = load_json(TRANSACTION_REPORT_PATH)
persisted_readback = load_json(READBACK_REPORT_PATH)
persisted_graph = load_json(REFERENCE_GRAPH_JSON)

required_metrics = {
    "unique_resource_count": (
        int(persisted_manifest.get("unique_resource_count", 0)),
        13,
    ),
    "transaction_bundle_count": (
        int(persisted_manifest.get("transaction_bundle_count", 0)),
        3,
    ),
    "local_validation_pass_rate": (
        float(
            persisted_local.get(
                "resource_validation_pass_rate",
                0,
            )
        ),
        1.0,
    ),
    "local_reference_integrity_rate": (
        float(
            persisted_local.get(
                "reference_integrity_rate",
                0,
            )
        ),
        1.0,
    ),
    "preliminary_observation_rate": (
        float(
            persisted_local.get(
                "preliminary_observation_rate",
                0,
            )
        ),
        1.0,
    ),
    "preliminary_report_rate": (
        float(
            persisted_local.get(
                "preliminary_report_rate",
                0,
            )
        ),
        1.0,
    ),
    "review_task_requested_rate": (
        float(
            persisted_local.get(
                "review_task_requested_rate",
                0,
            )
        ),
        1.0,
    ),
    "server_validation_pass_rate": (
        float(
            persisted_server.get(
                "validation_pass_rate",
                0,
            )
        ),
        1.0,
    ),
    "transaction_success_rate": (
        float(
            persisted_transaction.get(
                "transaction_success_rate",
                0,
            )
        ),
        1.0,
    ),
    "direct_read_success_rate": (
        float(
            persisted_readback.get(
                "direct_read_success_rate",
                0,
            )
        ),
        1.0,
    ),
    "critical_field_preservation_rate": (
        float(
            persisted_readback.get(
                "critical_field_preservation_rate",
                0,
            )
        ),
        1.0,
    ),
    "server_reference_integrity_rate": (
        float(
            persisted_graph.get(
                "reference_integrity_rate",
                0,
            )
        ),
        1.0,
    ),
}
failed_metrics = [
    f"{name}: {actual} != {expected}"
    for name, (actual, expected) in required_metrics.items()
    if actual != expected
]
if failed_metrics:
    raise AssertionError(
        "Notebook 07 completion metrics failed:\n"
        + "\n".join(f" - {item}" for item in failed_metrics)
    )

checksum_inventory = []
for path in sorted(
    {item.resolve() for item in required_artifacts},
    key=str,
):
    checksum_inventory.append(
        {
            "relative_path": path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

notebook_saved_in_drive = (
    NOTEBOOK_SAVE_PATH.exists()
    and NOTEBOOK_SAVE_PATH.stat().st_size > 0
)

final_audit = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "07",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": "completed",
    "audited_utc": utc_now(),
    "notebook_saved_in_drive": notebook_saved_in_drive,
    "server_base_url": FHIR_BASE_URL,
    "fhir_version": fhir_version,
    "metrics": {
        "case_count": 3,
        "unique_resource_count": 13,
        "device_count": 1,
        "observation_count": 3,
        "diagnostic_report_count": 3,
        "task_count": 3,
        "provenance_count": 3,
        "transaction_bundle_count": 3,
        "submitted_transaction_entry_count": int(
            persisted_transaction.get(
                "submitted_entry_count",
                0,
            )
        ),
        "local_validation_pass_rate": 1.0,
        "server_validation_pass_rate": 1.0,
        "local_reference_integrity_rate": 1.0,
        "server_reference_integrity_rate": 1.0,
        "transaction_success_rate": 1.0,
        "direct_read_success_rate": 1.0,
        "critical_field_preservation_rate": 1.0,
        "preliminary_observation_rate": 1.0,
        "preliminary_report_rate": 1.0,
        "review_task_requested_rate": 1.0,
        "autonomous_finalization_block_rate": 1.0,
    },
    "scope": {
        "current_ai_observations_created": True,
        "diagnostic_reports_created": True,
        "device_created": True,
        "algorithmic_provenance_created": True,
        "human_review_tasks_created": True,
        "transaction_writeback_performed": True,
        "server_readback_performed": True,
        "human_review_transition_executed": False,
        "accepted_results_created": False,
        "rejected_results_created": False,
        "correction_required_results_created": False,
    },
    "safety": {
        "synthetic_fhir_only": True,
        "public_deidentified_imaging_only": True,
        "all_ai_observations_preliminary": True,
        "all_diagnostic_reports_preliminary": True,
        "all_tasks_requested": True,
        "human_review_required_before_final": True,
        "reviewer_identity_invented": False,
        "autonomous_finalization_allowed": False,
        "clinical_validation_claimed": False,
        "clinical_deployment_claimed": False,
    },
    "output_paths": {
        "fhir_evidence_manifest": (
            FHIR_EVIDENCE_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "master_collection_bundle": (
            MASTER_EVIDENCE_BUNDLE_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "local_validation": LOCAL_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "server_validation": SERVER_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "transaction_report": (
            TRANSACTION_REPORT_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix()
        ),
        "readback_report": READBACK_REPORT_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "reference_graph": REFERENCE_GRAPH_JSON.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "network_log": NETWORK_LOG_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
    },
    "checksum_inventory": checksum_inventory,
    "next_notebook": (
        "08_NeuroFHIR_QC_Human_Review_Workflow.ipynb"
    ),
}
write_json(AUDIT_JSON_PATH, final_audit)

AUDIT_MD_PATH.write_text(
    textwrap.dedent(
        f"""
        # Notebook 07 — FHIR Evidence and Transaction Write-back

        **Status:** completed
        **Audited:** {final_audit['audited_utc']}
        **FHIR server:** {FHIR_BASE_URL}
        **FHIR version:** {fhir_version}

        ## Completed evidence

        - 13 unique FHIR R4 resources generated.
        - 3 preliminary AI-derived volume Observations.
        - 3 preliminary DiagnosticReports.
        - 3 requested human-review Tasks.
        - 3 algorithmic-generation Provenance resources.
        - 1 shared model/software Device.
        - 3 deterministic self-contained transaction Bundles.
        - 16/16 server validation targets passed (3 pre-write Bundles and 13 post-write resources).
        - 3/3 transactions succeeded.
        - {persisted_transaction['successful_entry_count']}/
          {persisted_transaction['submitted_entry_count']} transaction
          entries succeeded.
        - 13/13 resources read back.
        - Critical-field preservation passed at 100%.
        - Generated and source-context references resolved.

        ## Safety boundary

        Human review was not executed. No AI-generated Observation or
        DiagnosticReport became final. No reviewer identity was invented.
        Notebook 08 must execute and preserve the accept, reject and
        correction-required transitions.

        ## Next notebook

        `08_NeuroFHIR_QC_Human_Review_Workflow.ipynb`
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

nb07_entry["status"] = "completed"
nb07_entry["completed_utc"] = final_audit["audited_utc"]
nb07_entry["fhir_version"] = fhir_version
nb07_entry["unique_resource_count"] = 13
nb07_entry["transaction_bundle_count"] = 3
nb07_entry["server_validation_pass_rate"] = 1.0
nb07_entry["transaction_success_rate"] = 1.0
nb07_entry["critical_field_preservation_rate"] = 1.0
nb07_entry["audit_path"] = AUDIT_JSON_PATH.relative_to(
    PROJECT_ROOT
).as_posix()
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 104)
print("✅ Notebook 07 FHIR evidence and write-back completed")
print("✅ 13 unique linked FHIR R4 resources")
print("✅ Server validation, transaction write-back, and read-back passed")
print("✅ All AI results remain preliminary")
print("✅ Human-review decisions remain unexecuted")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print("📓 Manifest status: completed")
print("➡️ Notebook 08 — Human Review Workflow may begin")
print("=" * 104)

✅ Notebook 07 FHIR evidence and write-back completed
✅ 13 unique linked FHIR R4 resources
✅ Server validation, transaction write-back, and read-back passed
✅ All AI results remain preliminary
✅ Human-review decisions remain unexecuted
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_07_fhir_evidence_writeback_audit.json
📓 Manifest status: completed
➡️ Notebook 08 — Human Review Workflow may begin
